# [경쟁 제품 분석]

4개 제품별로 쿠팡에서 경쟁 제품 1~3개 선정 → 스펙 정리 → 리뷰 크롤링 → 소구점 분석
   
<br>   

- **185 커큐민+ (혜인서)** : 진입 고객 18,102명으로 최대 게이트웨이  
    → 신규 유입 감소 시 가장 타격 큼  
- **지니어스뉴 드롭스 계열 (그로우랩)** : 공구→자사몰 전환 1위 SKU  
    → 경쟁사 제품에 고객을 뺏기면 전환율에 직접 영향  
- **그로우뉴 (그로우랩)** : 후반기(25년 9월~) +422.9% 급성장 중  
    → 경쟁사 동향 파악이 성장 지속 여부에 중요  
- **투데이D3 계열 (파이토뉴트리)** : 자사몰 반복률이 가장 높은 제품(55.6%)  
    → 리텐션 핵심 제품의 경쟁 강도 확인 필요  

---

# [쿠팡]


- 분석 대상 4개 제품, 쿠팡 랭킹 페이지 (2026.06.15 오전 기준)  
    - 커큐민+:  
    https://www.coupang.com/np/search?component=&q=%EC%BB%A4%ED%81%90%EB%AF%BC&traceId=mqep4258&channel=user  
    - 지니어스뉴 / 어린이오메가3:  
    https://www.coupang.com/np/search?component=&q=%EC%96%B4%EB%A6%B0%EC%9D%B4+%EC%98%A4%EB%A9%94%EA%B0%803  
    - 그로우뉴 / 어린이칼마디:  
    https://www.coupang.com/np/search?component=&q=%EC%96%B4%EB%A6%B0%EC%9D%B4+%EC%B9%BC%EB%A7%88%EB%94%94  
    - 투데이D3 / 어린이비타민D:  
    https://www.coupang.com/np/search?component=&q=%EC%96%B4%EB%A6%B0%EC%9D%B4+%EB%B9%84%ED%83%80%EB%AF%BCd  

- 자동 크롤링으로 취합 불가하여 상기 페이지 내 랭킹 10위까지 수동 리스트 작성  
    - https://docs.google.com/spreadsheets/d/19QX4R-9Er-NGKRjqSfRCjj09TyQGVbeon190mlFIO_M/edit?usp=sharing  
    - 광고 제외 / 랭킹 등록되는 10위까지만 취합  
    - 용량만 다른 같은 제품이 중복 선정된 경우가 다수 있어 노트북으로 연결할 때는 중복 제거 실행  
    - 어린이칼마디 vs 어린이비타민D는 제품이 거의 동일하여, 그로우뉴와 투데이디3는 사실상 같은 경쟁 제품군과 경쇄하고 있음  

----

## 0. 경쟁 제품 선정

- 평점 20% + 리뷰수 40% + 랭킹 40%  

- 평점은 대부분 4.5~5.0에 몰려있어서 제품 간 차이를 거의 구분해주지 못하는 반면,  
리뷰수와 랭킹은 제품마다 값이 크게 갈려서 실제 어떤 제품이 더 많이 팔리고 검증됐는지를 훨씬 잘 보여줌  

- 그래서 변별력이 낮은 평점의 비중을 줄이고,  
변별력 있는 리뷰수·랭킹에 더 높은 비중을 주는 게 실제 시장 위치를 더 정확히 반영할 수 있을 것이라 판단  

In [61]:
!pip install selenium webdriver-manager beautifulsoup4 pandas konlpy
!pip install kiwipiepy

In [62]:
import pandas as pd
from collections import Counter


df = pd.read_excel('(쿠팡)_경쟁제품.xlsx', sheet_name='랭킹')

def select_competitors(df, top_n=3, min_rating=4.0, min_reviews=30):
    # 평점/리뷰수 기준 미달 제품 필터링
    filtered = df[(df['평점'] >= min_rating) & (df['리뷰수'] >= min_reviews)].copy()
    if filtered.empty:
        filtered = df.copy()

    # 점수 계산: 평점 20% + 리뷰수 40% + 랭킹 40%
    filtered['rating_score'] = filtered['평점'].rank(pct=True)
    filtered['review_score'] = filtered['리뷰수'].rank(pct=True)
    filtered['rank_score'] = 1 - filtered['랭킹'].rank(pct=True)

    filtered['score'] = (
        filtered['rating_score'] * 0.2
        + filtered['review_score'] * 0.4
        + filtered['rank_score'] * 0.4
    )

    filtered = filtered.sort_values('score', ascending=False)

    # 동일 제품(같은 회사+상품명, 용량만 다른 경우) 중복 제거
    filtered = filtered.drop_duplicates(subset=['회사', '상품명'], keep='first')

    return filtered.head(top_n).reset_index(drop=True)

# 분류별로 경쟁 제품 선정
selected_all = {}
for category in df['분류'].unique():
    sub = df[df['분류'] == category]
    selected = select_competitors(sub, top_n=3)
    selected_all[category] = selected
    print(f"\n=== {category} ===")
    print(selected[['score','상품명','회사','분량(정)','평점','리뷰수','가격','형태','랭킹']].to_string(index=False))


=== 커큐민 ===
 score              상품명   회사  분량(정)  평점  리뷰수    가격 형태  랭킹
  0.66 수용성 커큐민 바이오페린 맥스 올리트루     60 5.0  756 10900 알약   1
  0.66  페라큐민 강황 수용성 커큐민 가온담음    240 5.0 1996 19700 알약   3
  0.62      수용성 커큐민 맥시멈 순수채움    240 5.0 2351 28600 알약   6

=== 어린이오메가3 ===
   score                       상품명    회사  분량(정)  평점   리뷰수    가격  형태  랭킹
0.777778               키즈 츄어블 오메가3  세노비스    150 4.5 11967 26990 츄어블   1
0.744444 어린이 오메가 3 DHA 라즈베리 레몬맛 구미 릴크리터스    240 5.0  4890 39050 츄어블   2
0.522222              알티지 오메가3 츄어블  굿앤키즈    180 5.0  2239 29670 츄어블   4

=== 어린이칼마디 ===
   score                        상품명    회사  분량(정)  평점  리뷰수    가격  형태  랭킹
0.822222 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블  건국유업    240 4.5 3882 24900 츄어블   1
0.577778  젤튼튼 키즈 칼슘앤마그네슘 비타민D아연 츄어블   종근당    240 4.5  610 26320 츄어블   2
0.566667  금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D 비타민마을     60 5.0  381 10900 츄어블   3

=== 어린이비타민D ===
   score                        상품명    회사  분량(정)  평점  리뷰수    가격  형태  랭킹
0.755556 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블  건국유업    240 4.5 3882

## 1. 경쟁 제품 스펙 & 리뷰

In [63]:
# 선정된 경쟁 제품 목록 (평점/리뷰수/랭킹 기준 자동선정 결과)
selected_competitors = {
    "커큐민+": [
        {"name": "수용성 커큐민 바이오페인2X", "company": "센트휴", "rating": 5.0, "review_count": 3279, "price": 14700},
        {"name": "수용성 커큐민 맥시멈", "company": "순수채움", "rating": 5.0, "review_count": 2351, "price": 28600},
        {"name": "페라큐민 강황 수용성 커큐민", "company": "가온담음", "rating": 5.0, "review_count": 1996, "price": 19700},
    ],
    "지니어스뉴(오메가3)": [
        {"name": "어린이 오메가 3 DHA 라즈베리 레몬맛 구미", "company": "릴크리터스", "rating": 5.0, "review_count": 4890, "price": 39050},
        {"name": "키즈 츄어블 오메가3", "company": "세노비스", "rating": 4.5, "review_count": 11967, "price": 26990},
        {"name": "알티지 오메가3 츄어블", "company": "굿앤키즈", "rating": 5.0, "review_count": 2239, "price": 29670},
    ],
    "그로우뉴(칼마디)": [
        {"name": "쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블", "company": "건국유업", "rating": 4.5, "review_count": 3882, "price": 24900},
        {"name": "맘편한 어린이 칼슘 마그네슘 아연 비타민D", "company": "비타민마을", "rating": 5.0, "review_count": 1217, "price": 19900},
        {"name": "금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D", "company": "비타민마을", "rating": 5.0, "review_count": 381, "price": 10900},
    ],
    # 칼마디와 동일 경쟁 제품군 (참조용 - 위와 동일 리스트를 그대로 사용)
    "투데이D3(비타민D)": [
        {"name": "쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블", "company": "건국유업", "rating": 4.5, "review_count": 3882, "price": 24900},
        {"name": "맘편한 어린이 칼슘 마그네슘 아연 비타민D", "company": "비타민마을", "rating": 5.0, "review_count": 1217, "price": 19900},
        {"name": "금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D", "company": "비타민마을", "rating": 5.0, "review_count": 381, "price": 10900},
    ],
}

# 스펙 입력
product_specs_coupang = {
    # ---- 커큐민+ 경쟁 제품 ----
    "센트휴 - 수용성 커큐민 바이오페인2X": {
        "식품의 유형": "고형차",
        "생산자 및 소재지": "센트휴 국내제조 협력사",
        "원료명 및 함량": "수용성 강황추출물분말(인도산) 49.99%,치커리뿌리추출물분말(식이섬유 80% 이상/벨기에산),덱스트린,흑후추추출물(바이오페린 95% 이상/인도산),글루칸-30(덴마크산),캐롭분말 다크(스페인산),버섯혼합추출분말,영지버섯추출분말,강황추출분말,이산화규소,스테아린산마그네슘",
        "영양정보(1일 섭취량 기준)": "1일 1회 1~2정",
        "섭취방법": "물과 함께 섭취",
        "보관방법": "직사광선 피하여 서늘한 곳에 보관",
    },
    "순수채움 - 수용성 커큐민 맥시멈": {
        "식품의 유형": "음료베이스",
        "생산자 및 소재지": "(주)비에스바이오 /경기 안산시 단원구",
        "원료명 및 함량": "수용성 커큐민 추출분말(수용성 커큐민 추출물(인도),난소화성말토덱스트린),포도당,생강추출분말(국내산),27종 과일야채혼합분말(국내제조),보스웰리아추출물분말(인도),비타민C,퀘르세틴,흑후추추출분말",
        "영양정보(1일 섭취량 기준)": "1일 1회 1~2정",
        "섭취방법": "물과 함께 섭취",
        "보관방법": "직사광선 피하여 서늘한 곳에 보관",
    },
    "가온담음 - 페라큐민 강황 수용성 커큐민": {
        "식품의 유형": "음료베이스",
        "생산자 및 소재지": "가온담음 국내제조 협력사",
        "원료명 및 함량": "수용성 커큐민 추출분말(수용성 커큐민 추출분말[강황추출물분말(인도 산)(강황뿌리줄기) 난소화성말토덱스 트린],포도당,생강추출(국내산),27종과일야채혼합농축분말[세븐베리 농축액[블랙베리농축액(독일)] 야채 혼합농축액(독일산)],보스웰리아추출 물분말(인도산),비타민C,게르세틴,흑후추추출분말 토마토 함유",
        "영양정보(1일 섭취량 기준)": "1일 1회 1정",
        "섭취방법": "물과 함께 섭취",
        "보관방법": "직사광선 피하여 서늘한 곳에 보관",
    },

    # ---- 지니어스뉴(오메가3) 경쟁 제품 ----
    "릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미": {
        "식품의 유형": "기타가공식품",
        "생산자 및 소재지": "L'Il Critters, US",
        "원료명 및 함량": "USP 인증 / 미국 약전(United States Pharmacopeia)",
        "영양정보(1일 섭취량 기준)": "만 4세 이상 기준 1일 최대 2구미 섭취",
        "섭취방법": "완전히 씹어서 삼킬 수 있도록 지도해 주세요.",
        "보관방법": "직사광선 피하여 서늘한 곳에 보관",
    },
    "세노비스 - 키즈 츄어블 오메가3": {
        "식품의 유형": "건강기능식품",
        "생산자 및 소재지": "㈜서흥 /충청북도 청주시 흥덕구",
        "원료명 및 함량": "정제어유(정제어유,디-토코페롤혼합형/독일산), 비타민E(D-알파-토코페롤), 두나리엘라추출물(베타카로틴), 정제가공유지(말레이시아), 난소화성말토덱스트린, 밀납(네덜란드산/백납), 오렌지농출액분말, 자일리톨, 오렌지오일, 우유, 구연산, 트레할로스, DL-사과산, 효소처리스테비아, 비타민C, 대두레시틴, 수크랄로스(감미료) 우유,대두,돼지고기,고등어,밀 함유[캡슐기제]글리세린, 젤라틴(돈피), 변성전분, 에리스리톨, 식물성크림혼합분말, 글리세린지방산에스테르, 대두레시틴",
        "영양정보(1일 섭취량 기준)": "3~14세 1일 2회, 1회 3캡슐",
        "섭취방법": "충분히 씹어서 섭취하십시오",
        "보관방법": "직사광선을 피하여 습기가 적고 서늘한 곳에 보관",
    },
    "굿앤키즈 - 알티지 오메가3 츄어블": {
        "식품의 유형": "건강기능식품",
        "생산자 및 소재지": "코스맥스바이오(주) / 충청북도 제천시",
        "원료명 및 함량": "정제어유(정제어유, d-토코페롤(혼합형), 노르웨이산), d-α-토코페롤, 산화아연, 베타카로틴혼합제제{베타카로틴, 옥수수유, 비타민E(dl-α-토코페롤)}, 비타민D3혼합제제{비타민D3, 가공유지(팜유), 비타민E(dl-α-토코페롤)}, 포도씨유(스페인산), 밀납, 에리스리톨, 자일리톨, 오렌지향(천연향료), 오렌지향향료조제제(오렌지주스 덱스트린, 유당, 오렌지오일, 펙틴), 레몬향(천연향료), 요구르트향(향료), 구연산, 대두레시틴, 효소처리스테비아",
        "영양정보(1일 섭취량 기준)": "1일 3캡슐",
        "섭취방법": "꼭꼭 씹어서 먹기 / 캡슐을 잘라 내용물만 먹기 (그대로 먹기 or 음료에 타서 먹기)",
        "보관방법": "수분 및 열에 의해 영향을 받을 수 있으므로 직사광선을 피해 서늘한 곳에 보관",
    },

    # ---- 그로우뉴(칼마디) / 투데이D3(비타민D) 공통 경쟁 제품 ----
    "건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블": {
        "식품의 유형": "건강기능식품",
        "생산자 및 소재지": "(학)건국대학교 건국유업·건국햄 / 충청북도 음성군 대소면",
        "원료명 및 함량": "해조칼슘, 산화마그네슘, 산화아연, 비타민D3혼합제제(비타민D3, 설탕, 아라비아검, 옥수수전분, 중쇄중성지방유, 이산화규소, 비타민E), 엽산, 포도당, 자일리톨, 전지분유, 식물성크림분말, 밀크향분말(덱스트린, 프로필렌글리콜, 합성향료), 이산화규소, 스테아린산마그네슘, 효소처리스테비아, 크림향분말(덱스트린, 합성향료, 아라비아검, 프로필렌글리콜, 트리아세틴, 프로피온산), 과일채소혼합분말, 우유단백가수분해분말, 유산균혼합분말, 초유분말",
        "영양정보(1일 섭취량 기준)": "1일 2회, 1회 1정",
        "섭취방법": "씹어서 섭취",
        "보관방법": "직사광선 및 고온다습한 곳을 피하여 서늘한 곳에 보관",
    },
    "비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D": {
        "식품의 유형": "건강기능식품",
        "생산자 및 소재지": "엠에스바이오텍(주) / 충북 음성군 대소면",
        "원료명 및 함량": "꼬막칼슘분말, 산화마그네슘, 해조분말, 산화아연, 건조효모(비타민D3 함유), 바나나맛분말[포도당, 혼합제제(덱스트린, 향료, 아라비아검), 바나나추출분말, 이산화규소, 효소처리스테비아(감미료)], 정제포도당, 감미료(D-소비톨, 자일리톨, 자일리톨), 전지분유, 결정셀룰로스, 목화씨유분말, 혼합제제(덱스트린, 향료, 아라비아검), 효소처리스테비아(감미료), 혼합제제[포도당, 이산화규소, 합성향료, 프로필렌글리콜 함유], 무수구연산, 200만분의 1 혼합초유분말, 식물성유산균(균체), 홍삼농축액분말",
        "영양정보(1일 섭취량 기준)": "1일 2회, 1회 1정",
        "섭취방법": "씹어서 섭취",
        "보관방법": "직사광선을 피하여 습기가 적고 서늘한 곳에 보관",
    },
    "비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D": {
        "식품의 유형": "건강기능식품",
        "생산자 및 소재지": "엠에스바이오텍(주) / 충북 음성군 대소면",
        "원료명 및 함량": "꼬막칼슘분말, 산화마그네슘, 건조효모(아연), 건조효모분말(비타민D3함유), 정제포도당, 오렌지농축분말(덱스트린, 오렌지농축액, 아라비아검), D-소비톨, 말토덱스트린, 자일리톨, 결정셀룰로스, 혼합제제[오렌지쥬스, 덱스트린, 유당(우유), 천연오렌지오일, 펙틴], 대나무수액추출분말, MCT오일분말, 효소처리스테비아, 스테아린산마그네슘, 무수구연산, DL-사과산, 15종과일채소혼합분말, 우유 함유",
        "영양정보(1일 섭취량 기준)": "1일 1회, 1회 2정",
        "섭취방법": "씹어서 섭취",
        "보관방법": "직사광선을 피하여 습기가 적고 서늘한 곳에 보관",
    },
}

# 스펙 비교표 생성
spec_df = pd.DataFrame(product_specs_coupang).T
spec_df.index.name = "경쟁 제품"
spec_df

,식품의 유형,생산자 및 소재지,원료명 및 함량,영양정보(1일 섭취량 기준),섭취방법,보관방법
경쟁 제품,,,,,,
센트휴 - 수용성 커큐민 바이오페인2X,고형차,센트휴 국내제조 협력사,"수용성 강황추출물분말(인도산) 49.99%,치커리뿌리추출물분말(식이섬유 80% 이상...",1일 1회 1~2정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
순수채움 - 수용성 커큐민 맥시멈,음료베이스,(주)비에스바이오 /경기 안산시 단원구,"수용성 커큐민 추출분말(수용성 커큐민 추출물(인도),난소화성말토덱스트린),포도당,생...",1일 1회 1~2정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
가온담음 - 페라큐민 강황 수용성 커큐민,음료베이스,가온담음 국내제조 협력사,수용성 커큐민 추출분말(수용성 커큐민 추출분말[강황추출물분말(인도 산)(강황뿌리줄기...,1일 1회 1정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미,기타가공식품,"L'Il Critters, US",USP 인증 / 미국 약전(United States Pharmacopeia),만 4세 이상 기준 1일 최대 2구미 섭취,완전히 씹어서 삼킬 수 있도록 지도해 주세요.,직사광선 피하여 서늘한 곳에 보관
세노비스 - 키즈 츄어블 오메가3,건강기능식품,㈜서흥 /충청북도 청주시 흥덕구,"정제어유(정제어유,디-토코페롤혼합형/독일산), 비타민E(D-알파-토코페롤), 두나리...","3~14세 1일 2회, 1회 3캡슐",충분히 씹어서 섭취하십시오,직사광선을 피하여 습기가 적고 서늘한 곳에 보관
굿앤키즈 - 알티지 오메가3 츄어블,건강기능식품,코스맥스바이오(주) / 충청북도 제천시,"정제어유(정제어유, d-토코페롤(혼합형), 노르웨이산), d-α-토코페롤, 산화아연...",1일 3캡슐,꼭꼭 씹어서 먹기 / 캡슐을 잘라 내용물만 먹기 (그대로 먹기 or 음료에 타서 먹기),수분 및 열에 의해 영향을 받을 수 있으므로 직사광선을 피해 서늘한 곳에 보관
건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블,건강기능식품,(학)건국대학교 건국유업·건국햄 / 충청북도 음성군 대소면,"해조칼슘, 산화마그네슘, 산화아연, 비타민D3혼합제제(비타민D3, 설탕, 아라비아검...","1일 2회, 1회 1정",씹어서 섭취,직사광선 및 고온다습한 곳을 피하여 서늘한 곳에 보관
비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D,건강기능식품,엠에스바이오텍(주) / 충북 음성군 대소면,"꼬막칼슘분말, 산화마그네슘, 해조분말, 산화아연, 건조효모(비타민D3 함유), 바나...","1일 2회, 1회 1정",씹어서 섭취,직사광선을 피하여 습기가 적고 서늘한 곳에 보관
비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D,건강기능식품,엠에스바이오텍(주) / 충북 음성군 대소면,"꼬막칼슘분말, 산화마그네슘, 건조효모(아연), 건조효모분말(비타민D3함유), 정제포...","1일 1회, 1회 2정",씹어서 섭취,직사광선을 피하여 습기가 적고 서늘한 곳에 보관


### 1-1. 리뷰 수동 입력  
 
- 제품당 10개 (좋음 5개, 나쁨 5개)  
    나쁜 리뷰는 코멘트가 없는 경우가 많아 5개를 못채울 수 있음  
- 평점이 너무 좋은 리뷰만 있으면 편향될 수 있어, 낮은 별점도 같이 취합  
- 평점이 좋은 리뷰는 너무 길어서 전반적으로 축약된 버전으로 입력  

In [64]:
product_reviews_coupang = {
    "센트휴 - 수용성 커큐민 바이오페인2X": [
        "일반 커큐민보다 흡수율이 높은 수용성 커큐민이라 기대됨. 커큐민 함량 49.99%로 비교적 높은 편. 불필요한 첨가물을 줄인 점이 마음에 듦. 항염, 관절 건강, 항산화, 면역력 등에 도움을 기대하고 구매. 하루 1~2정으로 섭취가 간편함. 효과는 아직 개인차가 있을 수 있음. 과다 섭취 시 복통, 구토 등 위장 불편이 생길 수 있음. 철분 흡수를 방해할 수 있다는 점을 언급. 흡수율이 좋은 수용성 커큐민이라 염증 완화와 관절 건강에 도움이 될 것으로 기대되며, 원료와 함량이 만족스러워 꾸준히 복용해볼 만한 제품으로 평가함. 다만 효능은 개인차가 있고 과다 섭취는 주의가 필요함.",
        "수용성 커큐민 + 바이오페린(흑후추추출물) 조합으로 흡수율이 높다고 느낌. 정제 형태라 먹기 편하고 향이나 맛 부담이 적음. 꾸준히 섭취 후 아침 컨디션이 가벼워진 느낌. 항산화 및 건강 관리 목적으로 만족. 120정 대용량으로 가성비가 좋음. 효과는 개인 체감 위주이며 객관적인 변화는 확인하기 어려움. 흡수율을 높인 수용성 커큐민 제품으로, 섭취가 편하고 꾸준히 복용 시 컨디션 관리에 도움이 된다고 느낀 후기입니다. 대용량이라 가성비도 좋아 건강 관리를 위한 커큐민 제품을 찾는 사람에게 추천한다는 평가입니다.",
        "수용성 커큐민과 바이오페린으로 흡수율이 높다는 점이 매력적이고, 관절 및 몸의 뻐근함 관리 목적으로 구매. 2~4주 정도 꾸준히 섭취 후 아침 뻐근함 감소, 운동 후 회복이 조금 빨라진 느낌. 드라마틱한 변화보다는 꾸준히 먹을 이유가 있다고 체감. 비린 맛이나 강한 향이 거의 없음. 알약 크기가 적당해 먹기 편함. 120정으로 가성비가 괜찮음. 즉각적인 효과는 없음. 효과에 개인차가 있음. 관절·염증 관리와 일상적인 건강관리를 위해 꾸준히 섭취하기 좋은 커큐민 제품으로 평가하며, 재구매 의사가 있는 후기입니다.",
        "센트휴 수용성 커큐민 바이오페린 2X 120정 1개 주문후 배송 받았습니다. 몇년째 엄마의 병간호로 관절염이 생겨 진통제 복용보다 커큐민이 염증에 좋아하여 먹기 시작했습니다. 처음엔 효과가 있는지 모르겠더니 시나브로 지나면서 무릎통증이 덜해서 진통제는 거의 먹지 않고 커큐민을 꾸준히 8개월째 복용중입니다. 수용성이라 몸에 흡수가 잘된다고하여 꾸준히 챙겨 먹으니 효과를 보고 있어 저처럼 관절염 초기이신 분들은 독한 진통제 대신 수용성 커큐민을 드셔 보시기를 추천 드립니다. 좋은 제품 착한 가격에 감사합니다. 계속 재구매로 쟁여 둡니다.",
        "평소 고기, 당이 높은 음식들을 좋아하고 운동량이 적은 편이다보니 건강관리를 위해 영양제를 꾸준히 챙겨먹으려고 노력하는 편인데요. 요즘 ‘커큐민’이 현대인의 필수영양제로 유명하더라구요~ 골드박스에 특가로 떴길래 한번 꾸준히 먹어봐야겠다는 생각으로 구매하게되었습니다. 당이 높은 음식, 기름진음식을 좋아하면 몸에 염증이 생길 수 있는데 염증을 없애주는게 커큐민이라고 하더라구요~ 부가적으로 장내 가스제거에도 도움을 준다고 해서 3주간 먹어보았습니다. 가장 큰 변화는 예전에는 활동량이 증가하면 손이 잘 부었는데 그렇게 없어진거 같아요. 그리고 전보다 장이 답답한 것도 한결 좋아진 것을 느낄 수 있었습니다. 영양제의 중요한 점 중에 하나가 냄새라고 생각되는데 강황 특유의 냄새가 거의 안나서 냄새 민감하신 분들도 불편감 없이 복용하실 수 있을거 같아요. 크기도 크지않고 적당해서 어린학생들도 섭취 가능할 것 같습니다. 조금 더 복용해보고 후기를 올리겠습니다",
        "이름만 보고 커큐민추출물인줄 알았는데 아니고 강황추출물이었네요",
        "효능과 성분이 기대에 못 미쳐요",
        "커큐민이 좋다고 해서 또 다시 사서 먹기 시작. 한 이틀부터 얼굴에 빨게 지면서 열이 나네요. 강황이 몸 좋다고 하는데 왜 나는 부작용 날까요? 세계 10대 푸드 강황이라고 하는데요. 참 안타까워 유 속상해 며칠 있다가 괜찮아 지면 다시 한 번 먹어 봐야지 나이가 먹어서 그런가 자주 아프고 몸에 염증이 있다고 해서 커큐민 좋다고 하는데 정말 속상합니다",
    ],
    "순수채움 - 수용성 커큐민 맥시멈": [
        "수용성 커큐민이라 흡수율이 높을 것으로 기대하고 작은 알약을 선호하고, 하루 1정만 먹는 간편한 커큐민 제품을 찾다가 구매. 알약 크기가 작아 목 넘김이 편함. 하루 1정, 120정(약 4개월분)으로 복용이 간편함. 강한 냄새가 거의 없어 부담이 적음. 속 불편감 없이 섭취 가능. HACCP 인증 제품이라 신뢰감이 있음. 가성비가 좋음. 드라마틱한 변화는 없지만 꾸준히 건강관리를 한다는 만족감이 있음. 일반 커큐민보다 흡수에 대한 기대감이 있었음. 수용성 커큐민의 장점과 효능에 대한 설명이 더 자세했으면 좋겠음. 체감 효과는 크지 않음. 먹기 편한 작은 알약과 하루 1정의 간편함이 가장 큰 장점으로, 건강기능식품을 꾸준히 챙겨 먹기 어려운 사람에게 적합한 제품이라는 평가입니다. 효과는 완만하지만 부담 없이 장기 복용하기 좋고 재구매 의사도 있는 후기입니다.",
        "커큐민 함량(49.99%)이 높고 바이오페린이 포함되어 흡수율을 높인 점을 긍정적으로 평가하여 건강검진에서 높은 염증 수치를 확인한 후 염증 관리 목적으로 구매. 초반에는 큰 변화를 느끼지 못했으나 꾸준히 섭취하면서 효과를 체감. 침 피로감 감소, 손발·얼굴 붓기 완화, 전반적인 컨디션 개선을 경험. 1년 이상 복용 후 최근 검진에서 염증 수치가 정상 범위로 돌아왔다고 언급. 고함량 커큐민 + 바이오페린 조합이 좋고, 꾸준히 복용 시 염증 관리에 도움이 된다고 느낌. 자연스러운 향으로 거부감이 적음. 재구매 의사가 높음. 효과가 나타나기까지 시간이 필요함. 가루가 약간 날리고 섭취 후 손에 색이 묻을 수 있음. 염증 수치와 만성 피로, 붓기 관리 목적으로 장기 복용한 결과 만족도가 높은 후기입니다. 즉각적인 효과보다는 꾸준한 섭취 후 컨디션 개선과 염증 관리 효과를 체감했으며, 흡수율을 고려한 고함량 커큐민 제품이라는 점을 높게 평가했습니다.",
        "4개월분 대용량과 HACCP 인증으로 신뢰감이 있어서 수용성 커큐민이라 일반 커큐민보다 흡수율이 높다는 점에 관심을 갖고 구매. 일주일 정도 복용 후 아침에 몸이 조금 더 가볍고 피로감이 덜한 느낌. 속이 편하고 위장 부담이 적었다고 평가. 드라마틱한 변화보다는 꾸준히 복용할 가치가 있다고 느낌. 수용성 가공으로 흡수율 개선 기대. 위장 부담이 적은 편. 대용량으로 가성비가 좋음. 꾸준히 복용하기 편함. 효과 판단에는 시간이 필요함. 항응고제·항혈소판제 복용자는 상호작용 가능성이 있어 주의 필요. 식후 충분한 물과 함께 섭취 권장. 흡수율을 높인 수용성 커큐민 제품으로, 복용 후 아침 컨디션과 피로감 개선을 일부 체감했으며 속이 편한 점을 장점으로 꼽았습니다. 단기간에 큰 효과를 기대하기보다는 꾸준한 건강관리용으로 적합하다고 평가한 후기입니다.",
        "식약처 인증과 4개월분 대용량 구성에 만족했고, 기존에 복용하던 강황·커큐민 제품을 재구매하면서 선택. 수용성 커큐민이라 일반 커큐민보다 흡수율이 높다고 평가. 커큐민 외에도 피페린, 생강, 보스웰리아, 비타민C, 퀘르세틴 등이 함유되어 있음. 하루 건강관리용으로 꾸준히 섭취하기 좋음. 가성비가 좋고 장기 복용이 가능함. 위 건강 및 전반적인 건강관리를 위한 예방 목적으로 복용 중. 다양한 부원료가 포함되어 있어 만족도가 높음. 수용성 커큐민의 높은 흡수율과 다양한 부원료 구성을 장점으로 꼽으며, 건강관리와 예방 차원에서 꾸준히 복용하기 좋은 제품이라고 평가한 후기입니다. 대용량 구성과 가성비에도 만족하며 추천 의사를 밝혔습니다.",
        "식후 섭취 시 속이 편안함. 꾸준히 복용하면 전반적인 컨디션 관리에 도움이 되는 느낌. 염증 완화, 항산화, 관절 건강 관리 목적으로 기대. 식후 복용 시 위장 부담이 적음. 꾸준히 섭취하기 쉬움. 관절 및 염증 관리용으로 활용 가능. 공복 섭취 시 속이 불편할 수 있음. 제품마다 흡수율 차이가 있어 선택이 중요함. 혈액응고 관련 약물(항응고제 등) 복용자는 주의 필요. 커큐민은 특정 시간보다 식사와 함께 꾸준히 섭취하는 것이 중요하며, 지방이 포함된 음식과 함께 먹으면 흡수율을 높일 수 있다고 설명한 후기입니다. 식후 복용 시 속이 편하고 장기적인 건강관리와 관절·염증 관리에 도움이 될 수 있다고 평가했습니다.",
        "효능과 성분이 기대에 못 미쳐요",
        "1일 권장량에 함량이 못 미쳐요",
        "아직 먹고 있는데 그냥 저냥 잘 모르겠네요",
        "이틀째 먹고나서 부작용생겼어요. 이틀째 먹고나니 저랑 안맞는지 두드러기가 올라와요. 더이상 복용은 힘드네요~",
    ],
    "가온담음 - 페라큐민 강황 수용성 커큐민": [
        "건강검진에서 염증 수치가 높게 나와 만성 피로, 붓기 개선을 위해 복용 시작. 약사 추천과 수용성 커큐민의 흡수율을 보고 선택. 1년 이상 꾸준히 복용하며 염증 수치가 낮아졌다고 느낌. 아침 피로감, 얼굴·손발 붓기, 무기력감이 개선됨. 전반적인 컨디션과 활력이 좋아졌다고 평가. 수용성 커큐민이라 흡수율이 높다고 만족. 장기 복용에도 특별한 내성이나 불편함 없음. 재구매 의사가 매우 높음. 가루가 약간 묻어날 수 있음. 염증 수치 관리와 만성 피로, 붓기 개선을 위해 장기간 복용한 결과 만족도가 매우 높은 후기입니다. 꾸준히 섭취하면서 컨디션과 활력이 좋아졌다고 느꼈으며, 특히 중장년층의 건강관리에 도움이 된다고 평가했습니다. 다만 대부분의 효과는 개인 체감에 기반한 후기입니다.",
        "4개월분 대용량과 HACCP 인증 제품이라는 점을 긍정적으로 평가하여, 기존 커큐민 제품을 복용하던 중 흡수율이 높은 수용성 커큐민과 바이오페린 조합에 관심이 생겨 구매. 특별한 부작용 없이 복용 중. 전반적인 건강관리와 염증 관리 목적으로 만족하며 섭취. 알약 크기가 크지 않아 복용이 편하다고 느낌. 수용성 커큐민 + 바이오페린 조합으로 흡수율을 고려한 제품. 대용량이라 가성비가 좋음. 알약 크기가 적당해 섭취가 편리함. 재구매 의사가 있음. 직접 체감한 효과보다는 커큐민의 일반적인 효능과 성분 설명 위주의 후기. 효과에 대한 구체적인 경험은 제한적. 흡수율을 높인 수용성 커큐민과 바이오페린 조합, 대용량 구성, 편한 복용감을 장점으로 평가한 후기입니다. 전반적인 건강관리 목적의 꾸준한 섭취용으로 만족하며 재구매 의사를 밝히고 있습니다.",
        "수용성 커큐민과 바이오페린(피페린) 조합으로 흡수율을 높인 점을 가장 큰 장점으로 평가. 캡슐 형태라 먹기 편하고 강황 특유의 향이나 맛 부담이 적음. 속이 비교적 편하고 위장 부담이 적다는 후기가 많음. 관절 뻐근함, 피로감, 운동 후 회복 관리 목적으로 복용하는 경우가 많음. 일부는 아침 컨디션 개선이나 몸이 가벼워진 느낌을 언급. 즉각적인 효과보다는 몇 주~몇 달 동안 꾸준히 섭취하며 체감하는 편. 수용성 커큐민 + 바이오페린 조합. 복용 편의성이 좋음. HACCP 인증으로 신뢰감이 있음. 장기 건강관리용으로 적합. 효과에 개인차가 큼. 공복 섭취 시 속쓰림이나 더부룩함을 느낄 수 있음. 혈액응고 관련 약물 복용자는 주의 필요. 단기간에 큰 효과를 기대하면 실망할 수 있음. 흡수율을 높인 수용성 커큐민 제품으로, 관절·피로·염증 관리 목적의 장기 복용용 건강기능식품으로 평가됩니다. 복용 편의성과 흡수 설계에 대한 만족도가 높지만, 효과는 개인차가 크고 꾸준히 섭취해야 체감할 수 있다는 의견이 많습니다.",
        "몸이 무겁고 염증 관리가 필요하다고 느껴 구매. 오메가3, 비타민D와 함께 복용하면 좋다는 정보를 보고 선택. 즉각적인 효과보다는 꾸준히 복용해야 하는 제품이라고 느낌. 복용 후 몸이 조금 덜 무겁고 건강관리를 하는 느낌을 받음. 알 크기가 부담스럽지 않아 꾸준히 섭취하기 편함. 염증 관리용으로 기대할 수 있음. 4개월분 대용량으로 가성비가 좋음. 로켓배송 등 구매 편의성이 좋음. 장기 건강관리용으로 부담 없이 시작 가능. 강황 특유의 향이 약간 있음. 단기간에 눈에 띄는 효과는 기대하기 어려움. 염증 관리와 컨디션 관리를 위해 선택한 수용성 커큐민 제품으로, 즉각적인 변화보다는 꾸준히 복용하며 건강을 관리하는 용도로 만족한 후기입니다. 가성비와 복용 편의성이 좋고, 장기 복용용 건강관리 제품으로 추천한다는 평가입니다.",
        "40대 이후 피로감과 면역력 저하를 느끼면서 커큐민을 꾸준히 챙겨 먹게 됨. 커큐민의 항염·항산화 효과를 높게 평가하며 건강관리 필수 영양소로 인식. 피로 감소와 전반적인 건강관리에 도움이 될 것으로 기대. 현대인의 스트레스, 수면 부족, 운동 부족 등으로 인한 염증 관리에 유용하다고 생각. 항염, 항산화 기능에 대한 신뢰. 건강 유지와 면역 관리 목적에 적합. 중장년층 건강관리용으로 추천. 실제 복용 후 구체적인 체감 효과보다는 커큐민의 일반적인 효능 설명 비중이 큼. 피로감과 면역력 저하를 느끼는 중장년층에게 커큐민을 꾸준히 섭취할 것을 권하는 후기입니다. 항염·항산화 효과에 대한 기대가 크며, 전반적인 건강관리를 위한 필수 영양소로 평가하고 있습니다. 다만 실제 복용 경험보다는 커큐민의 일반적인 효능 소개가 중심인 후기입니다.",
        "구매 후기와 평점이 많아 선택했으나 기대와 달랐음. 복용 후 소변 색이 매우 진한 노란색(박카스 색) 으로 변했다고 언급. 복용 중 손에 카레가루 같은 노란 가루가 묻는 점도 이상하게 느꼈음. 커큐민 흡수율에 대한 의문을 제기하며 제품에 대한 만족도가 낮음. 복용 후 소변 색 변화와 제품 상태(가루 묻음)에 대한 불만을 제기한 후기입니다. 효과를 체감하지 못했고 흡수율에 대해서도 의구심을 표현하며 전반적으로 만족도가 낮은 평가를 남겼습니다.",
        "거의 다 먹어가는데 아직 효과는 모르겠어요",
        "효능이 기대에 못 미쳐요",
        "두통 주문해서 먹고 있는데 아직 효과를 모르겠어요",
        "효과 자체는 만족스러웠다고 평가. 피부에 있던 사마귀(또는 피부 돌기)가 줄어드는 변화를 경험. 피페린 성분에 대한 민감 반응이 있었던 것으로 추정. 복용 중 속쓰림이 발생해 섭취를 중단. 위가 약한 사람은 주의가 필요하다고 언급. 효과는 좋았지만 피페린 성분이 본인에게 맞지 않아 속쓰림이 발생한 후기입니다. 위장이 예민하거나 피페린에 민감한 사람은 휴지기를 두거나 섭취 여부를 신중히 결정하는 것이 좋겠다고 평가했습니다.",
    ],

    "릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미": [
        "아이 건강관리용으로 처음 구매한 오메가3로, 검사 후 추천받아 유산균·비타민D와 함께 섭취 중. 가격 부담이 적고 아이가 잘 먹는 점은 만족. 라즈베리·레몬 맛이라 비린내가 거의 없고 구미 형태라 먹기 편함. 다만 오메가3 함량(DHA 32mg)이 다소 낮아 함량을 중요하게 보는 경우 아쉬울 수 있음. 꾸준히 먹이는 데는 적합하지만, 다음에는 더 고함량 제품도 고려할 예정. 오메가3 향에 대한 호불호는 있을 수 있음.",
        "성장기 아이에게 먹이기 위해 처음 구매한 오메가3. 젤리(구미) 형태에 라즈베리·레몬 맛이라 비린내 부담이 적고 아이들이 간식처럼 잘 먹음. 알약을 어려워하는 아이에게 특히 적합. DHA·EPA가 포함되어 두뇌 발달과 눈 건강 관리를 기대하며 꾸준히 섭취 중. 아이가 스스로 찾을 정도로 맛 만족도가 높고 재구매 의사도 있음. 사용 연령 8세, 맛 만족도 높음, 효능 만족.",
        "아이가 생선이나 액상 오메가3를 잘 먹지 않아 DHA 보충용으로 구매. 라즈베리·레몬 맛의 구미 형태라 비린내가 거의 없고 간식처럼 잘 먹어 거부감이 적음. 물 없이도 먹을 수 있어 휴대와 섭취가 편리하며, 부모 입장에서도 챙겨주기 수월함. 식감도 적당히 쫀득해 아이가 부담 없이 섭취함. 다만 맛있어서 더 먹으려 할 수 있어 섭취량 관리가 필요하고, 여름철에는 구미가 말랑해질 수 있음. 전반적으로 아이가 꾸준히 잘 먹어 재구매 의사가 높은 제품.",
        "6세 아이가 처음 먹어본 오메가3인데 구미 형태라 거부감 없이 잘 먹고, 맛있다며 더 달라고 할 정도로 만족도가 높음. 라즈베리·레몬 맛이라 비린내가 거의 없고 식감도 적당해 섭취가 편함. 알약보다 먹이기 쉬워 부모 만족도도 높음. 아직 첫 복용이라 효과는 더 지켜봐야 하지만 첫인상은 매우 긍정적이며, 용량도 넉넉하고 배송·포장 상태도 만족스러웠다는 후기. 사용 연령 6세, 맛·만족도 모두 높음.",
        "8세 아이가 먹을 오메가3를 찾다가 구매. 라즈베리·레몬 맛의 구미 형태라 생선 비린내에 민감한 아이도 거부감 없이 잘 먹었고, 젤리처럼 부드러운 식감이라 섭취가 편함. DHA·EPA가 들어 있어 성장기 영양 보충용으로 만족하며, 하루 권장량 챙기기도 쉽고 휴대도 편리함. 배송과 제품 상태, 유통기한도 만족스러웠으며 아이가 간식처럼 잘 먹어 재구매 의사가 있는 후기. 사용 연령 8세, 맛·향·식감 만족도 높음.",
        "제품 효과보다 배송·포장 상태에 강한 불만을 표시한 후기. 완충재나 박스 없이 비닐 포장만 되어 있었고, 제품이 절반 정도 찌그러진 상태로 도착했다고 언급. 아이가 먹는 제품이라 더욱 불쾌했다고 하며, 포장 품질과 판매자 대응에 실망해 재구매 의사가 없다고 평가. 사용 연령 5세, 맛은 무난하지만 전반적인 만족도는 낮음.",
        "같은 브랜드 젤리는 잘 먹었지만, 이 DHA 구미는 아이가 잘 먹지 않아 아쉬웠다는 후기. 작성자도 직접 먹어봤는데 끝맛이 약간 쓰게 느껴졌다고 언급. 노랑·주황·보라색 3가지 맛이 들어 있으나 맛 구분은 어렵다고 평가. 전반적으로 맛에 대한 호불호가 있을 수 있으며, 효과는 보통 수준으로 만족한다고 평가. 사용 연령 4세.",
        "제품 자체보다는 품질 문제를 지적한 후기. 개봉 후 젤리 한 개의 상태가 이상했고 내부에 이물질처럼 보이는 것이 들어 있어 불안했다고 언급. 맛은 무난했지만 제품 품질에 대한 신뢰가 떨어졌으며, 기대했던 만족도에는 미치지 못했다고 평가. 사용 연령 4세.",
        "제품 효과보다는 유통기한이 너무 짧은 점에 불만을 제기한 후기. 6월에 구매했는데 유통기한이 10월까지로 약 4개월밖에 남지 않은 제품을 받았다고 언급. 판매 페이지의 유통기한 안내와 실제 수령 제품 간 차이가 있다고 느껴 아쉬움을 표현. 맛과 효능은 무난한 수준으로 평가했지만, 전반적인 만족도는 유통기한 문제로 낮아진 후기.",
        "품질 불량에 대한 강한 불만 후기. 3통을 구매했는데 모두 내용물이 녹아 서로 붙어 있어 꺼내 먹을 수 없는 상태였다고 언급. 아이가 먹는 제품인데 품질 관리가 제대로 되지 않았다고 비판하며 매우 낮은 만족도를 표현. 맛 자체는 무난하다고 평가했지만 제품 상태 문제로 사실상 재구매 의사가 없는 후기.",
    ],
    "세노비스 - 키즈 츄어블 오메가3": [
        "9세 아들은 잘 먹었지만 6세 딸은 한 번 먹고 더 먹기 싫어해 아이마다 호불호가 갈릴 수 있는 제품이라는 후기. 오렌지 맛과 향은 괜찮고 비린내도 거의 없었으나, 일반 젤리보다 질긴 껍질 같은 식감이 있어 아이들이 거부감을 느낄 수 있다고 평가. 작성자는 맛은 괜찮다고 느꼈고 브랜드 신뢰도도 높게 봤지만, 식감 때문에 아이 반응이 갈렸다고 언급. 전반적으로 초등학생 아들은 만족, 어린 아이는 호불호가 있을 수 있다는 의견. 사용 연령 9세, 맛은 무난, 효능 평가는 보통 수준.",
        "3회째 재구매 중인 제품으로 만족도가 매우 높은 후기. 물고기 모양의 츄어블 형태라 아이가 흥미를 보이며 스스로 찾을 정도로 잘 먹음. 오렌지 맛이 강하고 오메가3 특유의 비린내가 거의 없어 간식처럼 섭취 가능. EPA·DHA와 비타민, 베타카로틴까지 포함된 구성에 만족하며, 유통기한도 넉넉해 안심하고 구매했다고 평가. 먹이기 어려운 아이나 입맛이 까다로운 아이에게 특히 추천하며 재구매 의사가 매우 높음. 사용 연령 11세, 맛·효능 만족도 매우 높음.",
        "5세 아이가 꾸준히 먹을 수 있는 츄어블 오메가3로 만족한 후기. 캡슐 대신 씹어 먹는 형태라 부담이 적고 크기도 적당해 섭취가 편함. 아이가 거부감 없이 꾸준히 먹어 부모 입장에서 만족도가 높았으며, 대용량이라 오래 먹을 수 있고 보관도 편리하다고 평가. 다만 아이마다 입맛 차이가 있으므로 처음에는 소량으로 반응을 확인하는 것을 권장. 사용 연령 5세, 맛·효능 만족도 매우 높고 재구매 의향 있음.",
        "아이가 스스로 찾을 정도로 잘 먹는 오메가3 후기. DHA와 베타카로틴이 함유되어 두뇌·눈 건강을 기대하며 섭취 중이며, 물고기·거북이 등 바다친구 모양과 오렌지 맛 덕분에 아이가 즐겁게 먹음. 캡슐을 씹어 먹거나 안의 오렌지맛 내용물만 먹어도 되어 섭취가 편했고, 비린내가 거의 없다는 점도 장점으로 언급. 개봉 후에는 서늘하고 건조한 곳에 보관을 권장. 전반적으로 아이가 좋아해 재구매 의향이 있는 제품으로 평가.",
        "비린맛 때문에 오메가3를 거부하는 아이에게 적합한 제품이라는 후기. 츄어블 형태에 젤리 같은 식감이라 아이가 거부감 없이 잘 먹었고, 크기가 작아 씹어 먹기 편했다고 평가. 세노비스 브랜드에 대한 신뢰와 관리 편의성도 장점으로 언급. 다만 가격이 다소 있는 편이며, 맛있다고 과다 섭취하지 않도록 관리가 필요하다고 함. 성장기 아이의 두뇌·눈 건강 관리를 위해 추천하며, 간식처럼 인식시키면 섭취 습관 형성에 도움이 된다고 평가.",
        "아이가 비린맛 때문에 거부한 부정적 후기. 오렌지향으로 비린맛을 가리려 했지만 여전히 비린내와 맛이 강하게 느껴졌고, 아이가 먹자마자 거부했다고 평가. 젤리 껍질 식감도 질겨 만족스럽지 않았으며, 개봉 후 제품들이 서로 들러붙어 있는 점도 불만으로 언급. 세노비스 브랜드만 믿고 구매했지만 후기를 충분히 확인하지 않은 것을 후회했으며, 결국 다른 사람에게 주게 되었다고 함. 전반적으로 맛과 식감 때문에 재구매 의사가 없는 후기.",
        "유통기한이 매우 짧아 불만을 표시한 후기. 2개 묶음 할인으로 구매했는데 수령 후 확인해보니 유통기한이 약 2개월 남은 제품이라 실망했다고 함. 특히 오메가3는 산패 우려가 있어 어린아이에게 먹이기 불안하다고 언급. 판매 페이지에서 유통기한 임박 사실이 눈에 띄게 고지되지 않았다고 지적했으며, 비린내도 심하다고 평가. 전반적으로 구매를 후회하며 추천하지 않는다는 내용의 후기.",
        "맛과 식감에 대한 불만이 큰 부정적 후기. 젤리처럼 부드럽지 않고 껍질이 질겨 아이가 먹기 힘들어했으며, 섭취 후 피부 트러블이 발생했다고 언급. 지인 아이도 얼굴에 반응이 올라왔다고 해 제품에 대한 신뢰가 떨어졌다고 평가. 기존에 다른 오메가3나 비타민, 유산균은 문제없었는데 이 제품과 함께 먹은 비타민에서도 피부 반응이 생겼다고 느꼈으며, 결국 돈이 아깝고 재구매 의사가 없다는 내용의 후기.",
        "아이가 먹기 힘들어해 실망했다는 부정적 후기. 젤리 형태라 기대하고 구매했지만, 6세 아이가 섭취 후 토할 정도로 거부감을 보였고 삼키기 어렵다고 평가. 직접 먹어본 결과 겉부분이 질기고 씹히는 식감이 불편해 넘기기 힘들었다고 함. 브랜드를 믿고 구매했지만 맛과 식감 때문에 만족하지 못했으며, 개봉 후 반품도 어려워 아쉬움이 컸다는 내용의 후기.",
        "맛과 식감 때문에 강한 불만을 표현한 후기. 츄어블 제품으로 기대했지만 맛이 쓰고 비린 느낌이 강해 성인도 먹기 어렵다고 평가. 겉모습은 젤리 같지만 실제로는 질기고 건조한 식감이라 씹어 넘기기 힘들었으며, 안쪽 오렌지맛 내용물도 만족스럽지 않았다고 함. 원료(돼지 유래 젤라틴)를 보고 더 거부감이 생겼고, 아이가 먹기에는 적합하지 않다고 느꼈다고 언급. 전반적으로 제품 설명과 실제 섭취 경험의 차이가 커 매우 실망했다는 내용의 후기.",
    ],
    "굿앤키즈 - 알티지 오메가3 츄어블": [
        "생선을 잘 안 먹는 아이를 위해 선택한 츄어블 오메가3 후기. 비린맛과 캡슐 섭취를 어려워하는 아이를 위해 구매했으며, 깔끔한 포장과 아이가 좋아할 만한 디자인으로 첫인상이 좋았다고 평가. rTG 오메가3 제품으로 흡수율을 기대했고, 츄어블 형태라 물 없이 간편하게 먹을 수 있으며 비린 향이 거의 없어 아이가 거부감 없이 잘 먹었다고 함. 맛이 좋아 아이가 스스로 찾을 정도로 만족도가 높았고, 간식처럼 더 먹고 싶어 해 섭취량 관리가 필요하다고 언급. 전반적으로 꾸준히 먹이기 좋은 어린이 오메가3로 평가하며 재구매 의사가 있다고 밝힘.",
        "맛과 섭취 편의성이 좋다는 긍정적 후기. 다른 츄어블 오메가3의 비린내 때문에 제품을 바꿨는데, 이 제품은 비린내가 거의 없고 맛이 좋아 만족했다고 평가. 알약을 잘 못 삼키는 사람도 먹기 편하며 아이들에게도 추천할 만하다고 언급. EPA+DHA 500mg에 비타민D·E, 아연, 베타카로틴까지 포함되어 구성도 괜찮다고 봄. 캡슐 겉부분은 다소 질긴 편이지만 내용물만 먹어도 되어 큰 불편은 없었으며, 전반적으로 맛과 효능 모두 만족해 추천한다는 내용의 후기.",
        "8세 아이가 잘 먹어 재구매 중인 오메가3 후기. 비린내가 거의 없고 오렌지맛이라 아이가 거부감 없이 잘 먹으며, 입맛 까다로운 아이도 꾸준히 섭취해 만족한다고 평가. EPA+DHA 500mg과 비타민D·E, 아연, 베타카로틴이 함께 들어 있어 구성도 괜찮다고 언급. 하루 3캡슐 섭취로 관리가 쉽고 가격 부담도 크지 않아 장기 복용용으로 적합하다고 봄. 효과는 즉각 확인하기 어렵지만 집중력·눈 건강 관리를 기대하며 꾸준히 먹이고 있으며, 어린이용 오메가3를 찾는 부모에게 추천한다는 내용의 후기.",
        "비린맛 때문에 일반 오메가3를 못 먹던 성인이 만족한 후기. 큰 캡슐을 삼키기 어려워 오메가3를 꾸준히 먹지 못했는데, 이 제품은 츄어블 형태라 부담 없이 섭취 가능했다고 함. 오렌지맛이 먼저 느껴지고 비린맛이 거의 없어 간식처럼 먹기 편했으며, 젤리처럼 부드러운 식감도 장점으로 평가. 특히 오메가3 특유의 냄새와 맛 때문에 복용을 포기했던 사람에게 적합하다고 언급. 맛과 섭취 편의성이 좋아 꾸준히 먹을 수 있는 점을 가장 큰 장점으로 꼽았고, 큰 알약을 못 삼키거나 비린맛에 민감한 사람에게 추천한다는 내용의 후기.",
        "비린맛이 거의 없고 달콤한 맛 덕분에 아이가 거부감 없이 잘 먹는다는 재구매 후기. 생선이나 견과류를 잘 먹지 않는 아이의 오메가3 보충용으로 구매했으며, 이전에도 잘 먹어 재구매했다고 함. 캡슐이 크지 않고 적당히 씹어 먹는 츄어블 형태라 어린아이도 부담 없이 섭취 가능. 아침 식사 후 다른 영양제와 함께 챙겨주기 편하다고 평가. DHA가 성장기 아이의 두뇌 발달, 학습 및 집중력 향상에 도움이 될 것으로 기대하며 꾸준히 먹이고 있음. 전반적으로 맛, 식감, 섭취 편의성이 좋아 아이용 오메가3를 찾는 부모에게 무난하게 추천할 만한 제품이라는 내용.",
        "맛과 식감 때문에 섭취가 어렵다는 부정적 후기. 내용물을 주스에 타서 먹일 수 있다는 설명을 보고 구매했지만, 아이도 구역질을 하고 먹지 못했으며 성인인 작성자도 비린 맛 때문에 섭취가 힘들었다고 함. 캡슐은 고무 같은 맛이 나고, 내부 내용물은 맑은 오일이 아닌 걸쭉한 소스 같은 질감이라 주사기나 주스에 섞어 먹기도 불편했다고 평가. 기존 오메가3 제품은 거부감 없이 먹었지만 이 제품은 맛과 향이 맞지 않았으며, 어떻게 먹어야 할지 모르겠다는 의견. 전반적으로 비린 맛과 독특한 제형 때문에 만족도가 매우 낮았다는 내용의 후기.",
        "제품 간 품질 차이가 있는 것 같다는 불만 후기. 한 제품은 레몬향이 강해 아이에게 먹이기 괜찮았지만, 다른 제품은 비린내가 너무 심해 아이에게 주기 미안할 정도였다고 함. 제품마다 맛과 향 차이가 큰 것 같아 아쉽다는 의견이며, 아이가 먹는 제품인 만큼 품질 관리에 더 신경 써주길 바란다는 내용의 후기.",
        "식감 때문에 먹기 어렵다는 부정적 후기. 캡슐을 여러 번 씹어야 겨우 터질 정도로 질기며, 안에서 내용물이 나오기 전까지 오래 씹어야 해 매우 불편했다고 함. 식감이 마치 질긴 고무나 도루묵을 씹는 느낌과 비슷하다고 표현. 성인이 먹기에도 부담스러운 식감이라 아이들이 먹기에는 더 어려울 것 같다고 평가. 효능은 잘 모르겠지만, 씹는 질감 때문에 재구매 의사가 없다는 내용의 후기.",
        "비린 맛 때문에 아이들이 먹지 못했다는 부정적 후기. 9세와 7세 아이에게 먹이려 했지만 캡슐 크기가 크고, 씹자마자 비린 맛이 강하게 나서 바로 뱉었다고 함. 아이들이 거부감이 심해 결국 섭취하지 못했고, 남은 제품을 어떻게 처리해야 할지 난감하다는 의견. 전반적으로 맛과 크기 때문에 기대에 미치지 못했다는 내용의 후기.",
        "맛과 식감이 기대와 달라 실망했다는 후기. 구매평이 좋아 츄어블 제품이라 젤리 같은 식감을 기대했지만, 실제로는 씹으면 터지면서 걸쭉한 내용물이 나오고 캡슐은 고무를 씹는 듯한 질감이라고 평가. 맛도 기대에 못 미쳐 전반적으로 만족스럽지 않았으며, 일반적인 젤리형 영양제를 생각하고 구매했다면 실망할 수 있다는 내용의 후기.",
    ],

    "건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블": [
        "초등학생 딸을 위해 구매한 종합 성장 영양제에 대한 만족 후기. 알약을 삼키기 어려운 아이를 위해 츄어블 타입을 찾다가 선택했으며, 칼슘·마그네슘·아연·비타민D가 함께 들어 있어 여러 영양제를 따로 챙길 필요가 없는 점을 장점으로 꼽음. 아이가 맛있다며 잘 먹고 한 알 더 먹고 싶어 할 정도로 기호성이 좋았다고 함. 120정 대용량이라 가성비도 만족스럽고, 바쁜 아침에도 간편하게 챙길 수 있어 부모 입장에서 편리하다고 평가. 성장기 아이의 건강과 영양 보충을 위해 꾸준히 먹일 계획이며, 맛·영양 구성·가성비 모두 만족스러워 재구매 의사가 있다는 내용의 후기.",
        "성장기 아이 영양 관리를 위해 구매한 만족 후기. 장염 이후 체중이 줄고 또래보다 키 성장이 걱정되어 칼슘·마그네슘·아연·비타민D가 함께 들어 있는 제품을 선택했다고 함. 약 2달간 꾸준히 먹인 후 주변에서 키가 큰 것 같고 살도 붙어 보인다는 이야기를 들을 정도로 긍정적인 변화를 느꼈다고 평가. 정제 크기가 다소 큰 편이라 처음에는 걱정했지만, 아이가 간식처럼 잘 씹어 먹고 맛도 괜찮아 거부감 없이 섭취 중이라고 함. 영양제는 꾸준히 먹는 것이 중요한데 이 제품은 기호성이 좋아 챙겨주기 편하며, 120정 2개 구성으로 용량도 넉넉하고 가성비가 만족스럽다고 언급. 성장기 아이 영양제를 찾는 부모들에게 추천할 만한 제품이라는 내용의 후기.",
        "또래보다 키가 작은 아이의 성장 영양 관리를 위해 구매한 만족 후기. 의사 추천으로 영양제를 찾던 중 약국 제품보다 가성비가 좋아 선택했으며, 같은 가격에 2병 구성이라 만족했다고 함. 기존 제품과 맛이 달라 아이가 거부할까 걱정했지만, 우유맛이라 잘 먹고 스스로 챙겨 먹을 정도로 기호성이 좋았다고 평가. 아침저녁으로 꾸준히 섭취 중이며, 뚜껑이 안전 잠금 방식이라 아이가 혼자 열어 먹을 걱정이 없는 점도 장점으로 언급. 키 성장 자체를 단정할 수는 없지만 성장기 영양 보충에 도움이 되길 기대하며, 전반적으로 만족도가 높고 재구매 의향이 있다는 내용의 후기.",
        "칼슘·마그네슘 함량이 높아 만족한다는 후기. 여러 어린이 칼슘 영양제를 비교한 끝에 칼슘 300mg, 마그네슘 150mg, 비타민D를 함유한 점을 높게 평가해 구매했다고 함. 다른 제품들은 칼슘 함량이 200mg대 초반인 경우가 많았는데 이 제품은 함량이 상대적으로 높아 선택했다고 언급. 맛은 밀크맛으로 아이가 우유를 좋아하지 않아도 무난하게 먹을 수 있었으며, 한 번에 두 알을 주면 너무 달게 느껴져 아침·저녁으로 나눠 먹이고 있다고 함. 성분 구성과 가성비 모두 만족스럽고, 재구매 의사가 있을 정도로 추천할 만한 제품이라는 내용의 후기.",
        "마그네슘 보충을 위해 구매한 만족 후기. 둘째 아이가 잠을 깊게 자지 못하고 자주 깨는 모습 때문에 마그네슘 부족을 의심해 구매했다고 함. 이전에도 마그네슘을 먹였을 때 수면 상태가 좋아졌던 경험이 있었는데, 이번에도 섭취 후 다시 잠을 잘 자기 시작했다고 평가. 현재 6세, 8세 두 아이가 함께 먹고 있으며 아침·저녁으로 챙겨주고 있다고 함. 칼슘, 마그네슘, 아연, 비타민D를 한 번에 보충할 수 있어 편리하고, 츄어블 타입이라 아이들이 부담 없이 잘 먹는 점도 장점으로 언급. 용량도 넉넉하고 맛도 괜찮아 재구매 의사가 있으며, 성장기 아이 영양 관리용으로 만족스럽다는 내용의 후기.",
        "맛 때문에 아이가 먹기 힘들어했다는 부정적 후기. 10살 아이가 먹고 바로 뱉을 정도라 직접 먹어봤는데, 처음에는 우유맛이 나지만 씹을수록 단맛 뒤에 강한 쓴맛이 올라와 결국 뱉게 되었다고 함. 물로 헹구고 양치까지 해도 입안의 쓴맛이 오래 남았다고 평가. 아이가 먹기에는 끝맛이 너무 강하고 불쾌하게 느껴졌으며, 차라리 조금만 씹은 뒤 물과 함께 넘기는 방법을 추천한다고 언급. 맛 때문에 만족도가 낮았고, 개봉했지만 가능하다면 반품하고 싶을 정도로 아쉬웠다는 내용의 후기.",
        "강한 쓴맛 때문에 먹기 어렵다는 부정적 후기. 중학생 자녀가 어릴 때부터 다양한 칼슘제를 먹어봤지만, 이 제품은 먹은 뒤 남는 쓴맛이 너무 강해 힘들다고 평가. 후기에서 쓴맛 이야기를 보고도 설마 했는데 실제로 먹어보니 입안에 쓴맛이 오래 남아 불쾌했다고 함. 대용량이라 온 가족이 먹으려 했지만 맛 때문에 소비가 잘 되지 않는다고 언급. 칼슘 보충 효과 자체는 기대하지만, 기존에 먹던 제품들이 맛 면에서는 더 나았다고 평가했으며, 전반적으로 쓴맛 때문에 재구매 의사는 낮다는 내용의 후기.",
        "강한 쓴맛 때문에 실망했다는 부정적 후기. 좋은 후기를 믿고 구매했지만, 실제로 먹어보니 어린 감귤 껍질을 씹는 듯한 매우 강한 쓴맛이 났다고 평가. 쓴맛이 오래 남아 초콜릿을 먹어도 쉽게 사라지지 않았으며, 한 번 먹고 다시는 먹고 싶지 않을 정도였다고 함. 후기에서 맛있다고 한 사람들을 이해하기 어렵다고 표현할 만큼 맛에 대한 불만이 컸고, 결국 제품을 버릴 생각이라는 내용의 후기.",
        "처음에는 아이가 맛 때문에 거부했지만, 시간이 지나며 적응했다는 후기. 구매 초기에는 아이가 맛을 매우 싫어했고, 작성자 본인이 먹어봐도 선호하기 어려운 맛이라 아이가 먹지 않으면 의미가 없다고 평가. 하지만 약 2년이 지나 아이가 6살이 되면서는 무난하게 먹게 되었고, 달콤한 우유맛과 영양 성분 구성이 좋아 재구매했다고 함. 다만 어릴 때는 몇 번 먹이다가 거부하는 경우가 많았으며, 아이의 연령과 입맛에 따라 호불호가 크게 갈릴 수 있다는 내용의 후기.",
        "아이가 맛이 없어서 안 먹음. 작성자도 직접 먹어봤는데 텁텁한 맛이 강해 만족스럽지 않았다고 평가. 결국 가루로 만들어 약에 섞어 먹일 정도로 섭취가 어려웠으며, 생초코나 유산균 제품에 비해 기호성이 떨어진다고 언급. 전반적으로 맛 때문에 재구매 의사가 낮은 부정적 후기.",
    ],
    "비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D": [
        "아이 뼈 건강과 성장기 영양을 위해 선택. 칼슘뿐 아니라 마그네슘·아연·비타민D까지 함께 들어 있어 균형 잡힌 영양 구성이 장점이라고 평가. 180정 대용량이라 가성비가 좋고, GMP 인증 및 무첨가 콘셉트로 안심하고 구매. 알 크기가 크지 않고 향도 강하지 않아 아이가 거부감 없이 잘 먹는다고 함. 뼈 건강과 면역 관리까지 한 번에 챙길 수 있어 만족도가 높으며, 꾸준히 재구매 의사가 있는 긍정 후기.",
        "성장기 아이 영양 관리를 위해 꾸준히 재구매 중인 후기. 칼슘·마그네슘·아연·비타민D가 한 번에 들어 있어 성분 구성이 좋고, 무첨가 원료를 강조해 신뢰가 간다고 평가. 하루 2회 1정씩 섭취하는 방식이며 180정 대용량으로 가성비도 만족. 아이는 처음엔 바나나맛의 은은한 쓴맛 때문에 호불호가 있었지만, 오랫동안 먹으며 적응해 현재는 잘 섭취한다고 함. 초코맛 제품보다 성분이 깔끔하다고 느껴 선호하며, 안전한 성분과 합리적인 가격 때문에 계속 재구매 의사가 있다는 긍정 후기.",
        "기존에 먹이던 제품이 품절되어 대체품으로 구매. 아연이 포함된 칼슘·마그네슘·비타민D 조합이라 성분 구성이 마음에 들었고, 두 아이가 함께 먹기에도 부담 없는 가격이라 가성비를 높게 평가. 까다로운 둘째 아이도 맛있다며 잘 먹어 만족도가 높았음. 무첨가 부형제 공법으로 알약이 쉽게 부서질 수 있지만 품질 문제는 아니며 오히려 아이들이 먹기 편하다고 언급. 해조칼슘, 마그네슘, 아연, 비타민D 등 성장기 영양소를 한 번에 챙길 수 있어 전반적으로 매우 만족하는 긍정 후기.",
        "성장기 아이 영양 관리를 위해 꾸준히 재구매하는 후기. 칼슘·마그네슘·아연·비타민D를 한 번에 섭취할 수 있고, 무첨가 원료를 사용해 성분이 깔끔한 점을 높게 평가. 가격도 합리적이라 가성비가 좋다고 언급. 다만 맛은 달지 않고 약간 쓴 바나나맛이라 처음에는 아이들이 맛이 이상하다고 했지만, 블루베리 비타민과 함께 먹이며 적응해 지금은 잘 먹는다고 함. 성분과 안전성을 중시하는 부모에게 추천하며 전반적으로 매우 만족하는 긍정 후기.",
        "면역력이 약한 아이를 위해 구매한 후기. 칼슘·마그네슘·아연·비타민D를 한 번에 섭취할 수 있고 첨가물이 없는 점을 가장 큰 장점으로 평가. 부형제가 없어 알약이 잘 부서지고 손에 묻는 편이지만 오히려 성분이 깔끔하다는 점에서 긍정적으로 받아들임. 바나나맛이라 기존 제품보다 아이들이 더 잘 먹었고, 씹어 먹는 방식이라 섭취도 편하다고 언급. 가격도 만족스러워 가성비가 좋다고 평가하며, 당분간 꾸준히 먹여보고 재구매할 의사가 있다는 전반적으로 매우 만족한 후기.",
        "영양제가 배송 중 깨져 있는 경우가 많았고, 작성자는 품질 관리에 불만을 표시. 아이가 맛이 이상하다고 해서 직접 먹어봤는데 맛도 만족스럽지 않았다고 평가. 깨짐 문제와 기호성 모두 기대에 못 미쳐 실망했으며, 앞으로는 해당 브랜드 영양제를 구매하지 않겠다고 언급한 부정적인 후기.",
        "배송받은 제품에서 알약이 지나치게 많이 부서져 있어 불만을 제기한 후기. 한 통에 모아보니 상당한 양이 깨져 있었으며, 제품 자체 문제인지 포장·배송 문제인지 의문을 제기. 완충재(뽁뽁이) 등 포장 개선이 필요하다고 지적. 맛과 효능은 보통 수준으로 평가했지만, 반복되는 파손 문제 때문에 전반적인 만족도는 낮은 편이라는 내용.",
        "유통기한이 지나치게 짧아 불만을 제기한 후기. 7월에 구매했는데 유통기한이 다음 해 8월까지로, 180정을 하루 2정씩 먹으면 약 3개월이 필요한 제품임에도 남은 유통기한이 1년 남짓한 제품을 판매한 점을 문제 삼음. 특히 어린이용 영양제인 만큼 더 신경 써야 한다고 지적하며 판매 정책에 실망감을 표현. 맛과 효능 자체에는 만족했지만 유통기한 문제로 부정적인 평가를 남긴 후기.",
        "알약 크기가 커서 삼키기 어렵고, 씹어 먹기에도 맛이 좋지 않다고 평가한 후기. 작성자가 대신 먹어보다가 목에 걸릴 정도로 크기가 부담스러웠다고 언급. 아이가 먹기에는 불편해 보이며, 젤리 형태로 출시되면 좋겠다는 의견을 제시. 효능은 보통으로 평가했지만 크기와 기호성 문제로 만족도가 낮은 부정적 후기.",
        "아이가 맛이 없다고 거부해 먹이기 어려웠다는 후기. 다른 후기에서는 맛있다고 했지만 본인 아이는 입맛에 맞지 않아 권장량(2정)도 제대로 먹지 못했고, 아까워서 1정씩만 겨우 먹이고 있다고 함. 맛 때문에 만족도가 낮았으며, 기대했던 효과도 체감하지 못해 전반적으로 실망했다는 내용의 부정적 후기.",
    ],
    "비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D": [
        "어린이용 종합 영양제로 칼슘·마그네슘·아연·비타민D를 한 번에 섭취할 수 있어 만족한 후기. 평소 음식만으로 부족할 수 있는 영양소를 보충하기 위해 구매했으며, 아이들이 좋아하는 맛이라 거부감 없이 하루 2알씩 스스로 챙겨 먹는다고 함. 두부나 멸치 등을 잘 먹지 않는 아이의 칼슘 보충에 도움이 된다고 느꼈고, 유산균과 함께 꾸준히 먹이고 있다고 언급. 눈에 띄는 효과를 단정할 수는 없지만 기본 영양 관리용으로 신뢰하며, 여러 영양소를 한 번에 챙길 수 있는 점과 편의성에 만족해 추천하는 긍정 후기.",
        "지인 추천으로 구매한 어린이 칼슘·마그네슘·비타민D 영양제 후기. 칼슘, 마그네슘, 비타민D를 한 번에 챙길 수 있는 ‘칼마디’ 조합이라 편리하고, 하루 한 번 섭취하면 되어 관리가 수월하다고 평가. 처음에는 아이가 낯설어할까 걱정했지만 맛있다고 하며 스스로 챙겨 먹을 정도로 기호성이 좋았다고 함. 60정×3병 구성이라 꾸준히 먹이기 좋고 보관도 편리해 만족. 아이가 거부감 없이 먹는 성장기 영양제를 찾는 부모에게 추천하는 전반적으로 매우 긍정적인 후기.",
        "아이 키 성장 영양제를 찾다가 선택한 제품으로, 비싼 성장 전용 영양제보다 칼슘·마그네슘·비타민D 같은 기본 영양소를 챙기는 게 중요하다고 생각해 구매함. 가격이 합리적이라 꾸준히 먹이기 부담이 없고 가성비가 좋다고 평가. 칼슘, 마그네슘, 비타민D뿐 아니라 비타민B2, 아르기닌 등 성장기에 필요한 성분들이 골고루 들어 있어 구성에 만족한다고 함. 씹어 먹는 알약 형태에 아이들 비타민 특유의 달콤한 맛이라 아이가 거부감 없이 잘 먹고, 맛있다며 챙겨 먹을 정도라고 언급. 아직 복용 기간이 길지 않아 효과는 지켜보는 중이지만 밥을 조금 더 잘 먹는 느낌이 있고 전반적인 컨디션도 괜찮다고 평가. 오은영 프로그램 협찬 제품이라는 점에서 성분과 배합에 대한 신뢰감을 느꼈다고 하며, 한 통을 다 먹여본 뒤 성장 변화나 식습관 개선이 있으면 재구매와 추가 후기를 남길 계획이라고 밝힘. 전체적으로 가격, 성분 구성, 기호성 측면에서 만족도가 높은 초기 사용 후기.",
        "아이 성장과 건강을 위해 구매한 어린이 칼슘·마그네슘·비타민D 영양제로, 국내산 친환경 꼬막 칼슘과 아연, 비타민D가 함께 들어 있어 뼈·치아 형성, 면역 기능, 활력 관리에 도움이 될 것 같아 선택했다고 함. 두부·콩·멸치 등 칼슘 섭취가 부족한 아이나 야외활동이 적어 햇빛 노출이 부족한 아이에게 적합하다고 판단했으며, 오렌지맛이라 아이가 거부감 없이 잘 먹는 점을 장점으로 언급. 브랜드 인지도와 소비자 선호도 수상 이력도 신뢰 요소로 작용했다고 함. 칼슘은 뼈와 치아 건강, 마그네슘은 에너지 이용과 신경·근육 기능, 비타민D는 칼슘 흡수와 뼈 건강, 아연은 면역 기능과 세포분열에 도움을 주는 구성이라 만족한다고 평가. 전반적으로 성장기 어린이의 기본 영양 보충용으로 적합하고 가성비도 괜찮아 추천한다는 후기.",
        "6세 아이를 키우는 부모로서 편식이 심한 아이의 영양 불균형이 걱정돼 여러 제품을 비교한 끝에 구매했다고 함. 젤리형보다는 츄어블을 더 잘 먹는 아이 성향을 고려해 선택했으며, 딸기맛은 싫어하지만 오렌지맛은 잘 먹어 현재까지 거부감 없이 섭취 중이라고 평가. 크기는 다소 큰 편이지만 6세 아이가 씹어 먹기에는 무리가 없었고, 맛도 지나치게 강하지 않아 만족한다고 언급. 칼슘, 마그네슘, 비타민D, 아연이 한 번에 들어 있어 성장기 어린이의 뼈·치아 건강, 신경·근육 기능, 면역력 관리에 도움이 될 것 같다고 설명. 특히 국내산 친환경 꼬막 칼슘을 사용한 점과 칼슘 흡수를 고려한 마그네슘, 활성형 비타민D3 배합을 장점으로 꼽음. 두부·콩·멸치 섭취가 부족하거나 야외활동이 적은 아이에게 적합할 것 같다고 평가했으며, 아연 함유로 면역 기능 관리에도 도움이 될 것으로 기대. 하루 1회 2정 섭취 방식이라 챙겨주기 편하고, 무엇보다 아이가 스스로 찾아 먹을 정도로 기호성이 좋아 꾸준히 먹이기 좋다고 만족감을 표현. 전반적으로 성분 구성, 원료 신뢰도, 맛, 섭취 편의성 모두 만족스러워 성장기 어린이 영양제로 추천한다는 후기.",
        "오렌지 향이 난다고 해서 기대하고 구매했지만 실제로는 쌉쌀한 오렌지 껍질 같은 향과 맛이 강하게 느껴졌다고 함. 달콤한 맛보다는 특유의 쓴맛이 남아 아이가 바로 뱉어버렸으며, 여러 차례 먹여보려 했지만 결국 거부감이 심했다고 평가. 평소 잠에 예민해 마그네슘 보충을 위해 여러 제품을 시도해 왔고, 기존 밀크맛 제품은 아이가 싫어해 오렌지맛 제품으로 바꿨지만 만족스럽지 못했다고 언급. 쓴맛을 단맛으로 충분히 가리지 못한 점을 아쉬워하며, 오렌지맛을 기대하고 구매한다면 생각보다 호불호가 클 수 있다는 의견을 남김. 전체적으로 맛에 대한 불만이 크며 재구매 의사는 낮아 보이는 후기.",
        "다른 제품보다 가격이 비싼 편이라 기대하고 구매했지만, 아이가 한 입 먹자마자 너무 맛이 없다고 하며 뱉어버렸다고 함. 심지어 바로 양치까지 할 정도로 맛에 대한 거부감이 강했으며, 부모가 직접 먹어봐도 맛이 좋지 않았다고 평가. 성분이나 효능을 기대했지만 아이가 먹지 않으니 의미가 없다고 느꼈고, 결국 남은 한 통은 부모가 대신 먹어야 할 것 같다고 아쉬움을 표현. 전체적으로 가격 대비 만족도가 낮고, 특히 맛 때문에 재구매 의사가 없다는 부정적인 후기.",
        "아이가 맛이 없다고 하며 잘 먹지 않아 아쉬움을 남긴 후기입니다. 스스로 영양제를 챙겨 먹는 편이지만 이 제품은 기호성이 떨어져 꾸준히 섭취시키기 어렵다고 평가했습니다. 또한 제품 안에 들어 있는 습기제거제가 너무 큰 편이라 아이가 실수로 먹을까 봐 걱정된다고 언급했습니다. 맛에 대한 만족도가 낮아 아이의 반응이 좋지 않았고, 효능은 아직 특별히 체감하지 못해 보통 수준으로 평가했습니다. 전반적으로 맛과 포장 구성에 대한 아쉬움이 중심인 후기입니다.",
        "아이가 먹지 않아 부모가 직접 먹어봤지만 만족스럽지 못했다는 후기입니다. 조개껍질(해조칼슘) 원료 때문인지 껍질가루를 씹는 듯한 거칠고 텁텁한 식감이 느껴졌다고 평가했습니다. 맛과 식감 모두 어린아이가 먹기에는 부담스럽다고 생각했으며, 실제로 아이도 거부해 섭취를 하지 못했다고 합니다. 이미 제품을 개봉한 상태라 반품도 할 수 없어 더욱 아쉬웠다고 언급했습니다. 전반적으로 맛과 식감에 대한 불만이 크고, 어린아이에게는 추천하기 어렵다는 내용의 부정적인 후기입니다.",
        "아이가 잘 먹지 않는 이유가 먹다 보면 끝맛에서 약 같은 맛이 올라오기 때문이라는 후기입니다. 처음에는 먹을 수 있을 것 같지만 씹을수록 특유의 약맛이 느껴져 결국 거부하게 된다고 평가했습니다. 맛에 대한 만족도가 낮아 꾸준히 섭취하기 어렵다고 언급했으며, 효능 역시 특별히 체감하지 못해 보통 수준으로 평가했습니다. 전반적으로 기호성이 떨어져 아이가 잘 먹지 않는다는 점이 핵심인 부정적인 후기입니다.",
    ],
}

### 1-2. 스펙 비교표

In [65]:
from IPython.display import display

# ── 카테고리 매핑 (제품명 → 카테고리) ────────────────────────────
CATEGORY_MAP = {
    "센트휴 - 수용성 커큐민 바이오페인2X":              "커큐민+",
    "순수채움 - 수용성 커큐민 맥시멈":                  "커큐민+",
    "가온담음 - 페라큐민 강황 수용성 커큐민":            "커큐민+",
    "릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미": "지니어스뉴 (오메가3)",
    "세노비스 - 키즈 츄어블 오메가3":                   "지니어스뉴 (오메가3)",
    "굿앤키즈 - 알티지 오메가3 츄어블":                 "지니어스뉴 (오메가3)",
    "건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블": "그로우뉴 (칼마디)",
    "비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D":   "그로우뉴 (칼마디)",
    "비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D": "그로우뉴 (칼마디)",
}

CAT_COLORS = {
    "커큐민+":               "#D6E4F0",
    "지니어스뉴 (오메가3)":   "#E8F5E9",
    "그로우뉴 (칼마디)":      "#F3E5F5",
    "투데이D3 (비타민D)":     "#FFF8E1",
}

# ── product_specs_coupang → DataFrame 변환 ───────────────────────
df_coupang = pd.DataFrame(product_specs_coupang).T.reset_index()
df_coupang.columns = ["제품명", "식품유형", "생산자/소재지", "원료명 및 함량", "1일섭취량", "섭취방법", "보관방법"]
df_coupang.insert(0, "카테고리", df_coupang["제품명"].map(CATEGORY_MAP).fillna("기타"))

# 카테고리 순서 정렬
cat_order = ["커큐민+", "지니어스뉴 (오메가3)", "그로우뉴 (칼마디)", "투데이D3 (비타민D)", "기타"]
df_coupang["_cat_order"] = df_coupang["카테고리"].map({c: i for i, c in enumerate(cat_order)})
df_coupang = df_coupang.sort_values("_cat_order").drop(columns="_cat_order").reset_index(drop=True)

# ── 스타일 함수 ───────────────────────────────────────────────────
def color_row(row):
    bg = CAT_COLORS.get(row["카테고리"], "#FFFFFF")
    return [f"background-color: {bg}"] * len(row)

styled_coupang = (
    df_coupang.style
    .apply(color_row, axis=1)
    .set_properties(**{
        "font-size":      "12px",
        "font-family":    "Arial, sans-serif",
        "color":          "#111111",
        "text-align":     "left",
        "white-space":    "pre-wrap",
        "vertical-align": "top",
        "border":         "1px solid #d0d0d0",
        "padding":        "6px 8px",
    })
    .set_table_styles([
        {"selector": "thead th", "props": [
            ("background-color", "#2F5496"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("font-size", "12px"),
            ("text-align", "center"),
            ("padding", "8px"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "15px"),
            ("font-weight", "bold"),
            ("color", "#1F3864"),
            ("padding", "10px 0 6px 0"),
            ("text-align", "left"),
        ]},
    ])
    .set_caption("쿠팡 경쟁 제품 스펙 비교")
    .hide(axis="index")
)

display(styled_coupang)


카테고리,제품명,식품유형,생산자/소재지,원료명 및 함량,1일섭취량,섭취방법,보관방법
커큐민+,센트휴 - 수용성 커큐민 바이오페인2X,고형차,센트휴 국내제조 협력사,"수용성 강황추출물분말(인도산) 49.99%,치커리뿌리추출물분말(식이섬유 80% 이상/벨기에산),덱스트린,흑후추추출물(바이오페린 95% 이상/인도산),글루칸-30(덴마크산),캐롭분말 다크(스페인산),버섯혼합추출분말,영지버섯추출분말,강황추출분말,이산화규소,스테아린산마그네슘",1일 1회 1~2정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
커큐민+,순수채움 - 수용성 커큐민 맥시멈,음료베이스,(주)비에스바이오 /경기 안산시 단원구,"수용성 커큐민 추출분말(수용성 커큐민 추출물(인도),난소화성말토덱스트린),포도당,생강추출분말(국내산),27종 과일야채혼합분말(국내제조),보스웰리아추출물분말(인도),비타민C,퀘르세틴,흑후추추출분말",1일 1회 1~2정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
커큐민+,가온담음 - 페라큐민 강황 수용성 커큐민,음료베이스,가온담음 국내제조 협력사,"수용성 커큐민 추출분말(수용성 커큐민 추출분말[강황추출물분말(인도 산)(강황뿌리줄기) 난소화성말토덱스 트린],포도당,생강추출(국내산),27종과일야채혼합농축분말[세븐베리 농축액[블랙베리농축액(독일)] 야채 혼합농축액(독일산)],보스웰리아추출 물분말(인도산),비타민C,게르세틴,흑후추추출분말 토마토 함유",1일 1회 1정,물과 함께 섭취,직사광선 피하여 서늘한 곳에 보관
지니어스뉴 (오메가3),릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미,기타가공식품,"L'Il Critters, US",USP 인증 / 미국 약전(United States Pharmacopeia),만 4세 이상 기준 1일 최대 2구미 섭취,완전히 씹어서 삼킬 수 있도록 지도해 주세요.,직사광선 피하여 서늘한 곳에 보관
지니어스뉴 (오메가3),세노비스 - 키즈 츄어블 오메가3,건강기능식품,㈜서흥 /충청북도 청주시 흥덕구,"정제어유(정제어유,디-토코페롤혼합형/독일산), 비타민E(D-알파-토코페롤), 두나리엘라추출물(베타카로틴), 정제가공유지(말레이시아), 난소화성말토덱스트린, 밀납(네덜란드산/백납), 오렌지농출액분말, 자일리톨, 오렌지오일, 우유, 구연산, 트레할로스, DL-사과산, 효소처리스테비아, 비타민C, 대두레시틴, 수크랄로스(감미료) 우유,대두,돼지고기,고등어,밀 함유[캡슐기제]글리세린, 젤라틴(돈피), 변성전분, 에리스리톨, 식물성크림혼합분말, 글리세린지방산에스테르, 대두레시틴","3~14세 1일 2회, 1회 3캡슐",충분히 씹어서 섭취하십시오,직사광선을 피하여 습기가 적고 서늘한 곳에 보관
지니어스뉴 (오메가3),굿앤키즈 - 알티지 오메가3 츄어블,건강기능식품,코스맥스바이오(주) / 충청북도 제천시,"정제어유(정제어유, d-토코페롤(혼합형), 노르웨이산), d-α-토코페롤, 산화아연, 베타카로틴혼합제제{베타카로틴, 옥수수유, 비타민E(dl-α-토코페롤)}, 비타민D3혼합제제{비타민D3, 가공유지(팜유), 비타민E(dl-α-토코페롤)}, 포도씨유(스페인산), 밀납, 에리스리톨, 자일리톨, 오렌지향(천연향료), 오렌지향향료조제제(오렌지주스 덱스트린, 유당, 오렌지오일, 펙틴), 레몬향(천연향료), 요구르트향(향료), 구연산, 대두레시틴, 효소처리스테비아",1일 3캡슐,꼭꼭 씹어서 먹기 / 캡슐을 잘라 내용물만 먹기 (그대로 먹기 or 음료에 타서 먹기),수분 및 열에 의해 영향을 받을 수 있으므로 직사광선을 피해 서늘한 곳에 보관
그로우뉴 (칼마디),건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블,건강기능식품,(학)건국대학교 건국유업·건국햄 / 충청북도 음성군 대소면,"해조칼슘, 산화마그네슘, 산화아연, 비타민D3혼합제제(비타민D3, 설탕, 아라비아검, 옥수수전분, 중쇄중성지방유, 이산화규소, 비타민E), 엽산, 포도당, 자일리톨, 전지분유, 식물성크림분말, 밀크향분말(덱스트린, 프로필렌글리콜, 합성향료), 이산화규소, 스테아린산마그네슘, 효소처리스테비아, 크림향분말(덱스트린, 합성향료, 아라비아검, 프로필렌글리콜, 트리아세틴, 프로피온산), 과일채소혼합분말, 우유단백가수분해분말, 유산균혼합분말, 초유분말","1일 2회, 1회 1정",씹어서 섭취,직사광선 및 고온다습한 곳을 피하여 서늘한 곳에 보관
그로우뉴 (칼마디),비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D,건강기능식품,엠에스바이오텍(주) / 충북 음성군 대소면,"꼬막칼슘분말, 산화마그네슘, 해조분말, 산화아연, 건조효모(비타민D3 함유), 바나나맛분말[포도당, 혼합제제(덱스트린, 향료, 아라비아검), 바나나추출분말, 이산화규소, 효소처리스테비아(감미료)], 정제포도당, 감미료(D-소비톨, 자일리톨, 자일리톨), 전지분유, 결정셀룰로스, 목화씨유분말, 혼합제제(덱스트린, 향료, 아라비아검), 효소처리스테비아(감미료), 혼합제제[포도당, 이산화규소, 합성향료, 프로필렌글리콜 함유], 무수구연산, 200만분의 1 혼합초유분말, 식물성유산균(균체), 홍삼농축액분말","1일 2회, 1회 1정",씹어서 섭취,직사광선을 피하여 습기가 적고 서늘한 곳에 보관
그로우뉴 (칼마디),비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D,건강기능식품,엠에스바이오텍(주) / 충북 음성군 대소면,"꼬막칼슘분말, 산화마그네슘, 건조효모(아연), 건조효모분말(비타민D3함유), 정제포도당, 오렌지농축분말(덱스트린, 오렌지농축액, 아라비아검), D-소비톨, 말토덱스트린, 자일리톨, 결정셀룰로스, 혼합제제[오렌지쥬스, 덱스트린, 유당(우유), 천연오렌지오일, 펙틴], 대나무수액추출분말, MCT오일분말, 효소처리스테비아, 스테아린산마그네슘, 무수구연산, DL-사과산, 15종과일채소혼합분말, 우유 함유","1일 1회, 1회 2정",씹어서 섭취,직사광선을 피하여 습기가 적고 서늘한 곳에 보관


### 1-3. 소구점(Appeal Point) 키워드 분석

- Kiwi  
- 감정(긍정/부정) 톤 나누기  

In [84]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

def get_nouns(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag in ("NNG", "NNP")]

# 실제 리뷰 내용 바탕으로 카테고리 분류
appeal_keywords = {
    "효능/효과": ["효과", "성장", "면역", "건강", "도움", "변화", "체감", "흡수", "흡수율",
                "염증", "피로", "컨디션", "붓기", "관절"],   # "키" 제거
    "맛/복용편의": ["맛", "비린", "비린내", "거부감", "냄새", "젤리", "구미", "식감", "쓴맛",
                "삼키", "씹", "캡슐", "알약", "향"],
    "성분/안전성": ["성분", "함량", "원료", "인증", "무첨가", "안전", "부작용", "HACCP", "GMP"],
    "가격/가치": ["가격", "가성비", "비싸", "저렴", "대용량", "재구매", "용량"],
    "배송/포장/품질": ["배송", "포장", "파손", "유통기한", "품질", "이물질", "부서", "박스"],
}

stopwords = [
    "후기", "평가", "언급", "구매", "제품", "내용", "만족", "만족도",
    "전반", "정도", "대한", "수", "것", "후", "때문", "경우", "사용",
    "연령", "느낌", "부분", "전체", "기대", "표현", "느껴", "생각",
    "이상", "정", "점", "함",
]

HEIGHT_PATTERN = re.compile(r'(?<![가-힣])키(?:가|는|를|만|도)?(?=\s|[.,!?]|$)')

def tag_text_with_height_fix(text, pattern_map):
    tags = [tag for tag, pats in pattern_map.items() if any(p in text for p in pats)]
    if "효능/효과" not in tags and HEIGHT_PATTERN.search(text):
        tags.append("효능/효과")
    return tags

def analyze_appeal_points(reviews, top_k=20):
    if not reviews:
        return {}, []
    all_text = " ".join(reviews)
    nouns = get_nouns(all_text)
    nouns = [n for n in nouns if n not in stopwords and len(n) > 1] 
    freq = Counter(nouns)

    category_scores = {}
    for category, keywords in appeal_keywords.items():
        score = sum(freq.get(kw, 0) for kw in keywords)
        category_scores[category] = score

    return category_scores, freq.most_common(top_k)


appeal_results = {}
for product_key, reviews in product_reviews_coupang.items():
    scores, top_words = analyze_appeal_points(reviews)
    appeal_results[product_key] = {"category_scores": scores, "top_words": top_words}
    print(f"\n[{product_key}] (리뷰 {len(reviews)}개)")
    print("소구점 카테고리 점수:", scores)
    print("상위 키워드 Top 15:", top_words[:15])


print("\n\n===== 소구점 카테고리 비교표 =====")
appeal_score_df = pd.DataFrame(
    {k: v["category_scores"] for k, v in appeal_results.items()}
).T
print(appeal_score_df)

appeal_score_df.to_csv("appeal_score_result.csv", encoding="utf-8-sig")


[센트휴 - 수용성 커큐민 바이오페인2X] (리뷰 8개)
소구점 카테고리 점수: {'효능/효과': 40, '맛/복용편의': 4, '성분/안전성': 5, '가격/가치': 6, '배송/포장/품질': 2}
상위 키워드 Top 15: [('커큐민', 16), ('수용성', 8), ('섭취', 8), ('흡수', 7), ('관리', 7), ('건강', 6), ('효과', 6), ('염증', 6), ('복용', 6), ('관절', 4), ('도움', 4), ('강황', 4), ('개인차', 3), ('바이오페린', 3), ('추출물', 3)]

[순수채움 - 수용성 커큐민 맥시멈] (리뷰 9개)
소구점 카테고리 점수: {'효능/효과': 56, '맛/복용편의': 5, '성분/안전성': 9, '가격/가치': 8, '배송/포장/품질': 0}
상위 키워드 Top 15: [('복용', 19), ('관리', 14), ('흡수', 11), ('커큐민', 11), ('섭취', 10), ('건강', 10), ('염증', 9), ('효과', 8), ('수용성', 7), ('부담', 5), ('하루', 4), ('가능', 4), ('가성비', 4), ('장점', 4), ('체감', 4)]

[가온담음 - 페라큐민 강황 수용성 커큐민] (리뷰 10개)
소구점 카테고리 점수: {'효능/효과': 72, '맛/복용편의': 3, '성분/안전성': 6, '가격/가치': 7, '배송/포장/품질': 1}
상위 키워드 Top 15: [('복용', 25), ('관리', 21), ('효과', 17), ('커큐민', 15), ('건강', 14), ('섭취', 11), ('흡수', 10), ('염증', 9), ('수용성', 9), ('체감', 6), ('장기', 5), ('바이오페린', 5), ('조합', 5), ('목적', 5), ('피로', 4)]

[릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미] (리뷰 10개)
소구점 카테고리 점수: {'효능/효과': 6, '맛/복용편의': 25, '성분/

In [85]:
# ── 카테고리 ─────────────────────────
PRODUCT_CATEGORY_MAP = {
    # 쿠팡
    "센트휴 - 수용성 커큐민 바이오페인2X":                       "커큐민+",
    "순수채움 - 수용성 커큐민 맥시멈":                           "커큐민+",
    "가온담음 - 페라큐민 강황 수용성 커큐민":                     "커큐민+",
    "릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미":       "지니어스뉴 (오메가3)",
    "세노비스 - 키즈 츄어블 오메가3":                            "지니어스뉴 (오메가3)",
    "굿앤키즈 - 알티지 오메가3 츄어블":                          "지니어스뉴 (오메가3)",
    "건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블":    "그로우뉴 (칼마디)",
    "비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D":        "그로우뉴 (칼마디)",
    "비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D":    "그로우뉴 (칼마디)",
    # CJ (필요시 추가)
}

CAT_COLORS = {
    "커큐민+":               "#D6E4F0",
    "지니어스뉴 (오메가3)":   "#E8F5E9",
    "그로우뉴 (칼마디)":      "#F3E5F5",
    "투데이D3 (비타민D)":     "#FFF8E1",
}

# ── 데이터 준비 ──────────────────────────────────────────────────
df = sentiment_df.copy()
df["제품군"] = df["제품"].map(PRODUCT_CATEGORY_MAP).fillna("기타")
df["긍부정비율"] = df.apply(
    lambda r: round(r["긍정"] / (r["긍정"] + r["부정"]) * 100) if (r["긍정"] + r["부정"]) > 0 else None,
    axis=1
)

# ── 표 1: 제품 × 소구점 카테고리 (긍정/부정/중립 한 번에) ─────────
def make_combined_pivot(df):
    """제품 × 소구점으로 '긍정 N / 부정 N' 형태 피벗"""
    def fmt(row):
        pos, neg, neu = int(row["긍정"]), int(row["부정"]), int(row["중립"])
        return f"✅{pos}  ❌{neg}  🟡{neu}"

    df["요약"] = df.apply(fmt, axis=1)
    pivot = df.pivot(index="제품", columns="카테고리", values="요약").fillna("✅0  ❌0  🟡0")

    # 제품군 순서 정렬
    cat_order = ["커큐민+", "지니어스뉴 (오메가3)", "그로우뉴 (칼마디)"]
    pivot["_제품군"] = pivot.index.map(PRODUCT_CATEGORY_MAP).map(
        {c: i for i, c in enumerate(cat_order)}
    ).fillna(99)
    pivot = pivot.sort_values("_제품군").drop(columns="_제품군")
    return pivot

pivot_combined = make_combined_pivot(df)

def style_combined(pivot):
    def row_bg(row):
        제품군 = PRODUCT_CATEGORY_MAP.get(row.name, "기타")
        bg = CAT_COLORS.get(제품군, "#FFFFFF")
        return [f"background-color: {bg}"] * len(row)

    def cell_color(val):
        if isinstance(val, str) and "❌" in val:
            neg_count = int(val.split("❌")[1].split()[0])
            if neg_count >= 3:
                return "color: #C00000; font-weight: bold;"
        return "color: #111111;"

    return (
        pivot.style
        .apply(row_bg, axis=1)
        .map(cell_color)
        .set_properties(**{
            "font-size":      "11px",
            "font-family":    "Arial, sans-serif",
            "text-align":     "center",
            "vertical-align": "middle",
            "border":         "1px solid #d0d0d0",
            "padding":        "5px 8px",
            "white-space":    "nowrap",
        })
        .set_table_styles([
            {"selector": "thead th", "props": [
                ("background-color", "#2F5496"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("font-size", "11px"),
                ("text-align", "center"),
                ("padding", "7px"),
                ("white-space", "nowrap"),
            ]},
            {"selector": "th.row_heading", "props": [
                ("text-align", "left"),
                ("font-size", "11px"),
                ("padding", "5px 10px"),
                ("white-space", "nowrap"),
            ]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("color", "#1F3864"),
                ("padding", "10px 0 4px 0"),
                ("text-align", "left"),
            ]},
        ])
        .set_caption("소구점 카테고리별 감성 분포  (✅긍정  ❌부정  🟡중립) — 부정 3건 이상 빨간 강조")
    )

display(style_combined(pivot_combined))

print()

# ── 표 2: 제품군별 소구점 긍정 합산 히트맵 ──────────────────────
pivot_pos = df.pivot_table(index="제품군", columns="카테고리", values="긍정", aggfunc="sum").fillna(0)
pivot_neg = df.pivot_table(index="제품군", columns="카테고리", values="부정", aggfunc="sum").fillna(0)

cat_order = ["커큐민+", "지니어스뉴 (오메가3)", "그로우뉴 (칼마디)"]
for pv in [pivot_pos, pivot_neg]:
    pv.index = pd.CategoricalIndex(pv.index, categories=cat_order, ordered=True)
    pv.sort_index(inplace=True)

def heatmap_style(pivot, caption, color_low, color_high):
    max_val = pivot.values.max() if pivot.values.max() > 0 else 1

    def bg(val):
        if pd.isna(val) or val == 0:
            return "background-color: #F5F5F5; color: #AAAAAA;"
        ratio = val / max_val
        # low → high 색상 보간 (RGB)
        lr, lg, lb = int(color_low[1:3], 16), int(color_low[3:5], 16), int(color_low[5:7], 16)
        hr, hg, hb = int(color_high[1:3], 16), int(color_high[3:5], 16), int(color_high[5:7], 16)
        r = int(lr + (hr - lr) * ratio)
        g = int(lg + (hg - lg) * ratio)
        b = int(lb + (hb - lb) * ratio)
        text = "#FFFFFF" if ratio > 0.55 else "#111111"
        return f"background-color: rgb({r},{g},{b}); color: {text}; font-weight: {'bold' if ratio > 0.55 else 'normal'};"

    return (
        pivot.style
        .map(bg)
        .format(precision=0)
        .set_properties(**{
            "font-size":      "12px",
            "font-family":    "Arial, sans-serif",
            "text-align":     "center",
            "vertical-align": "middle",
            "border":         "1px solid #d0d0d0",
            "padding":        "6px 10px",
        })
        .set_table_styles([
            {"selector": "thead th", "props": [
                ("background-color", "#2F5496"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("font-size", "11px"),
                ("text-align", "center"),
                ("padding", "7px"),
            ]},
            {"selector": "th.row_heading", "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
                ("font-size", "11px"),
                ("padding", "6px 12px"),
            ]},
            {"selector": "table", "props": [("border-collapse", "collapse")]},
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("color", "#1F3864"),
                ("padding", "12px 0 4px 0"),
                ("text-align", "left"),
            ]},
        ])
        .set_caption(caption)
    )

display(heatmap_style(
    pivot_pos,
    "제품군 × 소구점  긍정 문장 수 히트맵  (진할수록 긍정 많음)",
    "#EBF5EB", "#1A7A1A"
))

display(heatmap_style(
    pivot_neg,
    "제품군 × 소구점  부정 문장 수 히트맵  (진할수록 불만 많음)",
    "#FFF0EE", "#C00000"
))

카테고리,가격/가치,맛/복용편의,배송/포장/품질,성분/안전성,효능/효과
제품,,,,,
가온담음 - 페라큐민 강황 수용성 커큐민,✅8 ❌0 🟡0,✅3 ❌0 🟡1,✅1 ❌0 🟡0,✅3 ❌1 🟡2,✅23 ❌2 🟡25
센트휴 - 수용성 커큐민 바이오페인2X,✅6 ❌0 🟡0,✅2 ❌2 🟡0,✅0 ❌0 🟡2,✅1 ❌0 🟡3,✅14 ❌0 🟡9
순수채움 - 수용성 커큐민 맥시멈,✅8 ❌0 🟡0,✅2 ❌1 🟡2,✅0 ❌0 🟡0,✅7 ❌1 🟡3,✅19 ❌0 🟡12
굿앤키즈 - 알티지 오메가3 츄어블,✅4 ❌0 🟡2,✅15 ❌9 🟡13,✅1 ❌1 🟡1,✅0 ❌0 🟡0,✅3 ❌0 🟡0
릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미,✅5 ❌2 🟡0,✅12 ❌3 🟡9,✅3 ❌7 🟡4,✅0 ❌1 🟡1,✅5 ❌2 🟡1
세노비스 - 키즈 츄어블 오메가3,✅6 ❌1 🟡1,✅11 ❌11 🟡8,✅1 ❌3 🟡0,✅0 ❌1 🟡0,✅2 ❌0 🟡0
건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블,✅8 ❌1 🟡2,✅10 ❌12 🟡7,✅0 ❌0 🟡0,✅4 ❌0 🟡1,✅7 ❌1 🟡2
비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D,✅4 ❌2 🟡1,✅12 ❌9 🟡4,✅0 ❌1 🟡0,✅4 ❌1 🟡1,✅14 ❌0 🟡2
비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D,✅8 ❌0 🟡1,✅8 ❌5 🟡4,✅1 ❌7 🟡2,✅5 ❌1 🟡2,✅4 ❌0 🟡3


카테고리,가격/가치,맛/복용편의,배송/포장/품질,성분/안전성,효능/효과
제품군,,,,,
커큐민+,22,7,1,11,56
지니어스뉴 (오메가3),15,38,5,0,10
그로우뉴 (칼마디),20,30,1,13,25


카테고리,가격/가치,맛/복용편의,배송/포장/품질,성분/안전성,효능/효과
제품군,,,,,
커큐민+,0,3,0,2,2
지니어스뉴 (오메가3),3,23,11,2,2
그로우뉴 (칼마디),3,26,8,2,1


- 커큐민 3종 (효능/효과 40~72)  
    - 공통 : '흡수', '염증', '관절', '복용'이 핵심 키워드  
    - 가온담음(72점) : '바이오페린', '조합', '목적', '피로', '장기'가 상위권  
        → 복합 성분 + 장기 복용 컨셉이 강함  
    - 순수채움 : 성분/안전성(9)·가격/가치(8)가 다른 두 제품보다 높고  
        '부담', '가성비', '장점'이 top_words에 들어있어 "저자극+가성비" 포지셔닝  
    - 센트휴 : '강황', '추출물'이 top_words에 있음  

- 오메가3 3종 (맛/복용편의 25~32)    
    - 릴크리터스 : 배송/포장/품질이 14점으로 압도적  
        top_words에 '상태', '유통', '기한', '품질'이 모두 들어있어  
        제품력보다 포장/유통 관리 문제가 핵심 약점임이 명확  
    - 세노비스 : '식감'(10), '거부감'(5), '불만'(4), '반응'(피부반응)이 top_words에 있어  
        식감(질긴 껍질)으로 인한 거부감이 핵심 약점  
    - 굿앤키즈 : 점수가 가장 높지만(32) '추천', '내용물', '편의'도 함께 보여  
        맛/식감 이슈가 있지만 동시에 만족 후기도 섞여 있는 양면적 상황  

- 칼마디/비타민D 3종  
    - 건국유업 : '쓴맛'이 9점으로 top_words 중 가장 두드러진 단일 키워드 ("쓴맛 후味"가 명확한 약점)  
    - 비타민마을-맘편한 : 성분/안전성(12)·가격/가치(11)·배송품질(9)이 골고루 높고  
        '문제', '첨가'가 top_words에 있는데 "무첨가/성분" 포지셔닝과 동시에 "파손 문제" 불만이 공존   
    - 비타민마을-금쪽같은 내새끼 : 효능/효과가 24점으로 어린이 제품 중 유일하게 높은데,  
        '추천', '오렌지', '구성', '기호'가 함께 나와 맛과 효능 모두 비교적 균형 잡힌 포지셔닝으로 보임  

        
<br>  

> 분석 한계 — 키워드 기반 소구점 분류의 정밀도  
>  
> 본 소구점 분석은 사전 정의한 키워드 매칭 방식(rule-based)으로, 정교한 자연어 의미 분석이 아닙니다.   
> 특히 짧은 한 글자 키워드는 다른 단어의 일부로 우연히 매칭되는 위험이 있습니다.  
>  
> 예시: 초기 분석에서 "키"를 효능/효과(키 성장) 키워드로 사용했으나, "삼키다/삼키기"  
> (섭취 편의성을 의미)의 부분 문자열로도 매칭되어 일부 맛/복용편의 관련 문장이 효능/효과로  
> 잘못 집계되는 문제가 발견되었습니다. (검증 결과 노이즈 매칭 6건 vs 정상 매칭 5건으로,  
> 거의 1:1 수준의 오염도)  
> 
> 이를 정규식 기반 단어 경계 체크로 보정했으며, 이후에도 유사한 1~2글자 키워드는  
> 추가 키워드 도입 시 같은 방식의 substring 충돌 검증이 필요합니다.  
> 전반적으로 본 분석의 숫자는 정밀한 감성 점수가 아니라 카테고리별 상대적 비중과 톤 파악용 참고 지표로 해석해야 합니다.  

In [86]:
import re

def get_nouns(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag in ("NNG", "NNP")]


# ----------------------------------------------------------------
# 감정(긍정/부정) 판별용 단어 사전
# → 문장 단위로 긍정 단어 수 vs 부정 단어 수를 비교해서 판별
#   (정교한 감성분석은 아니지만, 카테고리별 톤 파악에는 충분)
# ----------------------------------------------------------------
positive_words = [
    "만족", "좋", "추천", "효과적", "도움", "개선", "재구매", "잘 먹", "잘먹",
    "편하", "신뢰", "기대", "장점", "깔끔", "넉넉", "적당", "괜찮", "잘 섭취",
    "거부감 없", "거부감 적", "흡수율이 높", "맛있", "간식처럼",
]

negative_words = [
    "불만", "실망", "아쉬", "거부", "부작용", "별로", "못 먹", "안 먹",
    "쓴맛", "비린", "불편", "환불", "반품", "후회", "문제", "파손", "부서",
    "짧", "낮", "걸렸", "포기", "거부했", "거부한", "토할", "뱉", "텁텁",
    "맛이 없", "맛없", "기대에 못 미", "재구매 의사가 없", "재구매 의사는 낮",
]


def split_sentences(text):
    # 마침표/느낌표/물음표 뒤 공백, 또는 줄바꿈 기준으로 문장 분리
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


def sentence_sentiment(sentence):
    pos = sum(sentence.count(w) for w in positive_words)
    neg = sum(sentence.count(w) for w in negative_words)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    else:
        return "neutral"


def analyze_appeal_sentiment(reviews):
    """카테고리별 (positive, negative, neutral) 문장 수 집계"""
    category_sentiment = {
        cat: {"positive": 0, "negative": 0, "neutral": 0} for cat in appeal_keywords
    }

    for review in reviews:
        for sent in split_sentences(review):
            for cat, keywords in appeal_keywords.items():
                if any(kw in sent for kw in keywords):     # ← 바로 이 줄
                    sentiment = sentence_sentiment(sent)
                    category_sentiment[cat][sentiment] += 1

    return category_sentiment



sentiment_results = {}
for product_key, reviews in product_reviews_coupang.items():
    sentiment_results[product_key] = analyze_appeal_sentiment(reviews)

# ----------------------------------------------------------------
# 결과를 표로 정리 (카테고리별 긍정/부정/중립 문장 수)
# ----------------------------------------------------------------
rows = []
for product_key, cat_sentiments in sentiment_results.items():
    for cat, counts in cat_sentiments.items():
        rows.append({
            "제품": product_key,
            "카테고리": cat,
            "긍정": counts["positive"],
            "부정": counts["negative"],
            "중립": counts["neutral"],
        })

sentiment_df = pd.DataFrame(rows)
pivot_pos = sentiment_df.pivot(index="제품", columns="카테고리", values="긍정")
pivot_neg = sentiment_df.pivot(index="제품", columns="카테고리", values="부정")

print("===== 카테고리별 긍정 문장 수 =====")
print(pivot_pos)
print("\n===== 카테고리별 부정 문장 수 =====")
print(pivot_neg)

sentiment_df.to_csv("appeal_sentiment_result.csv", index=False, encoding="utf-8-sig")

===== 카테고리별 긍정 문장 수 =====
카테고리                               가격/가치  맛/복용편의  배송/포장/품질  성분/안전성  효능/효과
제품                                                                       
가온담음 - 페라큐민 강황 수용성 커큐민                 8       3         1       3     23
건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블      8      10         0       4      7
굿앤키즈 - 알티지 오메가3 츄어블                    4      15         1       0      3
릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미      5      12         3       0      5
비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D      4      12         0       4     14
비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D        8       8         1       5      4
세노비스 - 키즈 츄어블 오메가3                     6      11         1       0      2
센트휴 - 수용성 커큐민 바이오페인2X                  6       2         0       1     14
순수채움 - 수용성 커큐민 맥시멈                     8       2         0       7     19

===== 카테고리별 부정 문장 수 =====
카테고리                               가격/가치  맛/복용편의  배송/포장/품질  성분/안전성  효능/효과
제품                                                         

- Kiwi로 확인한 인사이트와 동일함 확인  

---
---

# [CJ온스타일]

- 분석 대상 4개 제품, CJ온스타일 / 인기순 (2026.06.15 오후 기준)  
    - 커큐민+ / 커큐민영양제:  
    https://display.cjonstyle.com/p/search/searchAllList?k=%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D
    - 지니어스뉴 / 어린이오메가3:  
    https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%98%A4%EB%A9%94%EA%B0%803&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%98%A4%EB%A9%94%EA%B0%803%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22scrollTop%22%3A0%2C%22isSortChanged%22%3Atrue%7D  
    - 그로우뉴 / 어린이칼마디:  
    https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%B9%BC%EB%A7%88%EB%94%94&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%B9%BC%EB%A7%88%EB%94%94%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D 
    - 투데이D3 / 어린이비타민D:  
    https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%20%EB%B9%84%ED%83%80%EB%AF%BCd&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%20%EB%B9%84%ED%83%80%EB%AF%BCd%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D  

- 자동 크롤링  
    - 인기순으로 상위 15위까지만 취합  
    - 용량만 다른 같은 제품이 중복 선정된 경우가 다수 있어 노트북으로 연결할 때는 중복 제거 실행  
    - CJ온스타일에서는 어린이칼마디 vs 어린이비타민D는 제품이 겹치는 경쟁사가 전혀 없음 (쿠팡과 다름)  
    - CJ온스타일은 별점 후기만 있고, 텍스트 후기는 없어서 스펙 기반 비교표만 정리  
    

---

## 0. 경쟁 제품 선정 - 크롤링(20분소요)

In [69]:
# CJ온스타일 - 4개 제품 인기순 상위 15개 수집

# import 및 드라이버 세팅
import time

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup


def get_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    return driver


# 4개 제품의 CJ온스타일 검색 URL (인기순 = BEST_SELLING_DESC)
target_products = {
    "커큐민+": "https://display.cjonstyle.com/p/search/searchAllList?k=%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D",
    "지니어스뉴_어린이오메가3": "https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%98%A4%EB%A9%94%EA%B0%803&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%98%A4%EB%A9%94%EA%B0%803%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22scrollTop%22%3A0%2C%22isSortChanged%22%3Atrue%7D",
    "그로우뉴_어린이칼마디": "https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%B9%BC%EB%A7%88%EB%94%94&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%EC%B9%BC%EB%A7%88%EB%94%94%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D",
    "투데이D3_어린이비타민D": "https://display.cjonstyle.com/p/search/searchAllList?k=%EC%96%B4%EB%A6%B0%EC%9D%B4%20%EB%B9%84%ED%83%80%EB%AF%BCd&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%96%B4%EB%A6%B0%EC%9D%B4%20%EB%B9%84%ED%83%80%EB%AF%BCd%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D",
}


# [디버그] 먼저 한 개 URL로 정상 로딩되는지 확인
test_url = target_products["커큐민+"]

driver = get_driver()
driver.get(test_url)
time.sleep(5)  # SPA 렌더링 + API 호출 대기

print("페이지 제목:", driver.title)
print("현재 URL:", driver.current_url)

with open("debug_cjonstyle.html", "w", encoding="utf-8") as f:
    f.write(driver.page_source)

driver.quit()

페이지 제목: CJ온스타일 검색결과
현재 URL: https://display.cjonstyle.com/p/search/searchAllList?k=%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C&searchType=ALL#%7B%22t%22%3A%22API%22%2C%22isMobile%22%3Afalse%2C%22o%22%3A%22BEST_SELLING_DESC%22%2C%22k%22%3A%22%EC%BB%A4%ED%81%90%EB%AF%BC%EC%98%81%EC%96%91%EC%A0%9C%22%2C%22of%22%3A0%2C%22s%22%3A48%2C%22sc%22%3A%22%22%2C%22firstSearch%22%3Afalse%2C%22listingType%22%3A2%2C%22og%22%3Afalse%2C%22chn%22%3A%2230002002%22%2C%22isSortChanged%22%3Atrue%7D


In [70]:
# 상품 카드 영역의 class명 F12로 확인
# (이 class명을 4번 셀의 셀렉터에 반영해야 함)


# 검색 결과 -> 상품 리스트 크롤링 (상위 15개)
def crawl_cjonstyle_list(url, top_n=15, sleep=5):
    driver = get_driver()
    driver.get(url)
    time.sleep(sleep)  # SPA 렌더링 대기

    # 추가 상품 로드를 위해 스크롤 (필요 시)
    for _ in range(2):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    # 상품 카드: <a class="gaclass" href="https://display.cjonstyle.com/p/item/...">
    items = soup.select("a.gaclass[href*='/p/item/']")

    results = []
    for idx, item in enumerate(items[:top_n], start=1):
        name_tag = item.select_one("strong.goods_name")
        price_tag = item.select_one("span.wrap_price ins span.num")
        review_tag = item.select_one("strong.customer_count span.num")

        name = name_tag.get_text(strip=True) if name_tag else None
        price_text = price_tag.get_text(strip=True) if price_tag else None
        review_text = review_tag.get_text(strip=True) if review_tag else "0"
        review_count = re.sub(r"[^\d]", "", review_text)
        link = item.get("href")

        results.append(
            {
                "natural_rank": idx,  # 인기순(BEST_SELLING_DESC) 정렬 기준 순위
                "name": name,
                "price": price_text,
                "review_count": int(review_count) if review_count else 0,
                # ⚠️ 평점은 리스트 화면에 없음. 상세페이지에서 별도 수집 필요
                "url": link,
            }
        )

    return pd.DataFrame(results)

# 4개 제품 전체 수집
product_candidates_cj = {}
for product_name, url in target_products.items():
    print(f"[수집 중] {product_name}")
    df = crawl_cjonstyle_list(url, top_n=15)
    product_candidates_cj[product_name] = df
    time.sleep(3)

for name, df in product_candidates_cj.items():
    print(f"\n=== {name} ===")
    print(df)

# 상위 15개 상품의 상세페이지에서 평점 가져오기
import json

def crawl_cjonstyle_rating(url, sleep=3):
    driver = get_driver()
    driver.get(url)
    time.sleep(sleep)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
        except (TypeError, json.JSONDecodeError):
            continue

        candidates = data if isinstance(data, list) else [data]
        for d in candidates:
            agg = d.get("aggregateRating")
            if agg:
                return agg.get("ratingValue"), agg.get("reviewCount")

    return None, None


for product_name, df in product_candidates_cj.items():
    ratings = []
    review_counts_jsonld = []
    for _, row in df.iterrows():
        print(f"[평점 수집] {product_name} - {row['name']}")
        rating, review_count_jsonld = crawl_cjonstyle_rating(row["url"])
        ratings.append(rating)
        review_counts_jsonld.append(review_count_jsonld)
        time.sleep(2)
    df["rating"] = ratings
    df["review_count_jsonld"] = review_counts_jsonld

for name, df in product_candidates_cj.items():
    print(f"\n=== {name} ===")
    print(df[["natural_rank", "name", "rating", "review_count", "review_count_jsonld", "price"]])

[수집 중] 커큐민+
[수집 중] 지니어스뉴_어린이오메가3
[수집 중] 그로우뉴_어린이칼마디
[수집 중] 투데이D3_어린이비타민D

=== 커큐민+ ===
    natural_rank                                               name    price  \
0              1              [동가게PICK]SOVS 인퓨즈드워터 3박스+골든부스트 커큐민1박스   99,900   
1              2                          닥터린 하이퍼셀 수용성 커큐민 맥스 x 5박스   89,930   
2              3            더리틀스 레스베라 커큐민부스터 메리바 파이토솜 커큐민 2박스(2개월분)   94,900   
3              4                 안국건강 수용성 리포좀 커큐민 강황 플러스 60정 (1개월분)   19,350   
4              5  해외배송 Nature Made 네이쳐메이드 터메릭 커큐민 60정 Turmeric C...   47,000   
5              6                              조은약초 강황환 스틱형 3g x 30포   10,900   
6              7                          오허브 강황가루 강황분말 강황밥 커큐민 1kg   16,500   
7              8                                갑당약초 진도 울금/강황환 300g   23,900   
8              9                          닥터린 하이퍼셀 수용성 커큐민 맥스 x 3박스   56,950   
9             10  GNM 자연의품격 GNM 수용성 리포좀 커큐민 플러스 강황 30정 X 5박스(총 5...   67,360   
10            11             안국건강

In [71]:
for name, df in product_candidates_cj.items():
    print(f"\n=== {name} ===")
    print(df[["natural_rank", "name", "rating", "review_count", "price"]])


=== 커큐민+ ===
    natural_rank                                               name  rating  \
0              1              [동가게PICK]SOVS 인퓨즈드워터 3박스+골든부스트 커큐민1박스     4.8   
1              2                          닥터린 하이퍼셀 수용성 커큐민 맥스 x 5박스     4.9   
2              3            더리틀스 레스베라 커큐민부스터 메리바 파이토솜 커큐민 2박스(2개월분)     5.0   
3              4                 안국건강 수용성 리포좀 커큐민 강황 플러스 60정 (1개월분)     NaN   
4              5  해외배송 Nature Made 네이쳐메이드 터메릭 커큐민 60정 Turmeric C...     NaN   
5              6                              조은약초 강황환 스틱형 3g x 30포     NaN   
6              7                          오허브 강황가루 강황분말 강황밥 커큐민 1kg     4.6   
7              8                                갑당약초 진도 울금/강황환 300g     5.0   
8              9                          닥터린 하이퍼셀 수용성 커큐민 맥스 x 3박스     4.7   
9             10  GNM 자연의품격 GNM 수용성 리포좀 커큐민 플러스 강황 30정 X 5박스(총 5...     4.5   
10            11             안국건강 수용성 리포좀 커큐민 강황 플러스 60정 3박스 (3개월분)     4.5   
11            12                      

In [72]:
# 경쟁 제품 선정
# 쿠팡과 달리 리뷰수가 0~10건대가 대부분이라 min_reviews 기준을 거의 두지 않음
# 평점이 있으면 20%, 없으면 리뷰수/랭킹만으로 점수화
def select_competitors_cj(df, top_n=3, min_reviews=0):
    filtered = df[df['review_count'] >= min_reviews].copy()
    if filtered.empty:
        filtered = df.copy()

    filtered['review_score'] = filtered['review_count'].rank(pct=True)
    filtered['rank_score'] = 1 - filtered['natural_rank'].rank(pct=True)

    if 'rating' in filtered.columns and filtered['rating'].notna().any():
        filtered['rating_score'] = filtered['rating'].fillna(0).rank(pct=True)
        filtered['score'] = (
            filtered['rating_score'] * 0.2
            + filtered['review_score'] * 0.4
            + filtered['rank_score'] * 0.4
        )
    else:
        filtered['score'] = (
            filtered['review_score'] * 0.5
            + filtered['rank_score'] * 0.5
        )

    filtered = filtered.sort_values('score', ascending=False)

    # 용량만 다른 동일 제품 중복 제거 (상품명 앞 2단어 = 브랜드 키 기준)
    filtered['brand_key'] = filtered['name'].apply(lambda x: ' '.join(x.split()[:2]))
    filtered = filtered.drop_duplicates(subset=['brand_key'], keep='first')

    return filtered.head(top_n).reset_index(drop=True)


selected_all_cj = {}
for category, df in product_candidates_cj.items():
    selected = select_competitors_cj(df, top_n=3)
    selected_all_cj[category] = selected
    print(f"\n=== {category} ===")
    print(selected[['natural_rank', 'name', 'rating', 'review_count', 'price', 'score']])


=== 커큐민+ ===
   natural_rank                                     name  rating  \
0             1    [동가게PICK]SOVS 인퓨즈드워터 3박스+골든부스트 커큐민1박스     4.8   
1             2                닥터린 하이퍼셀 수용성 커큐민 맥스 x 5박스     4.9   
2             3  더리틀스 레스베라 커큐민부스터 메리바 파이토솜 커큐민 2박스(2개월분)     5.0   

   review_count   price     score  
0           147  99,900  0.906667  
1            13  89,930  0.840000  
2             3  94,900  0.713333  

=== 지니어스뉴_어린이오메가3 ===
   natural_rank                                               name  rating  \
0             2  휴럼 뮴 키즈 스마트 오메가3 구미 2박스 어린이 아기 츄어블 오메가3 과일맛 영양 젤리     5.0   
1             1                         네추럴라이즈 키즈 오메가3 꾸미 3박스 3개월분     5.0   
2             5  내츄럴플러스 어린이 알티지 오메가3 츄어블 오렌지맛 90캡슐 1통 / 베타카로틴 비...     4.4   

   review_count   price     score  
0             2  99,000  0.826667  
1             1  41,710  0.786667  
2             7  17,900  0.773333  

=== 그로우뉴_어린이칼마디 ===
   natural_rank                                               name  

---

## 1. 경쟁 제품 스펙 & 별점

In [73]:
from IPython.display import display, HTML

# ================================================================
# CJ온스타일 경쟁 제품 스펙 입력 템플릿
# 각 제품 상세페이지 > 상품정보제공고시 표를 보고 빈 칸을 채워주세요
# ================================================================
product_specs_cj = {
    # ---- 커큐민+ ----
    "[동가게PICK]SOVS 인퓨즈드워터 3박스+골든부스트 커큐민1박스": {
        "식품의 유형": "액상차",
        "원료명 및 함량": "정제수,바나바잎혼합추출액[고형분 0.05% 이상,바나바잎(인도네시아산),볶은돼지감자(국산),바나바잎추출물(인도산)],천연향료(유기농레몬추출물) 0.1%,탄산수소나트륨 ",
        "영양정보(1일 섭취량 기준)": "코로솔산으로서 0.45~1.3mg / 1병 750ml",
        "섭취방법": "마셔서 섭취",
        "주의사항": "균형 잡힌 식생활 권장.본제품은 질병의 예방·치료를 위한 제품이 아닙니다.",
    },
    "닥터린 하이퍼셀 수용성 커큐민 맥스 x 5박스": {
        "식품의 유형": "음료베이스",
        "원료명 및 함량": "수용성커큐민추출물(인도산/강황추출물분말, 말토덱스트린),정제포도당,꼬막칼슘(국산),목화씨유분말(미국산),대나무수액추출물분말,옥수수단백추출물,흑후추추출물,비타민C,비타민E혼합제제(DL-α-토코페릴아세테이트, 포도당시럽분말, 변성전분, 이산화규소),건조효모(아연),케르세틴,베타글루칸,호박추출분말,버섯혼합추출분말",
        "영양정보(1일 섭취량 기준)": "1일 1회, 1회 1정",
        "섭취방법": "충분한 물과 함께 섭취",
        "주의사항": "직사광선과 고온다습한 곳을 피하여 실온에 보관하십시오.",
    },
    "오허브 강황가루 강황분말 강황밥 커큐민 1kg": {
        "식품의 유형": "고형차",
        "원료명 및 함량": "강황뿌리줄기 100%(인도산)",
        "영양정보(1일 섭취량 기준)": "1일 2~3회, 1회 2~3g",
        "섭취방법": "물에 섞어 섭취 또는 기호에 맞게 요리나 음료와 함께 섭취",
        "주의사항": "직사광선을 피하고 서늘한 곳에 보관. 장시간 보관 시 냉장 보관",
    },

    # ---- 지니어스뉴(오메가3) ----
    "휴럼 뮴 키즈 스마트 오메가3 구미 2박스": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "정제어유(멸치),자일리톨,정제수,D-소비톨,젤라틴(어류),구연산삼나트륨,천연향료(투티프루티향),아라비아검,DL-α-토코페롤,비타민C,스테비올배당체,파프리카추출색소",
        "영양정보(1일 섭취량 기준)": "1일 2정",
        "섭취방법": "씹어서 섭취",
        "주의사항": "직사광선을 피해 서늘한 곳에 보관. 개인에 따라 피부 관련 이상반응이 발생할 수 있음.",
    },
    "네추럴라이즈 키즈 오메가3 꾸미 3박스 3개월분": {
        "식품의 유형": "캔디류",
        "원료명 및 함량": "설탕,물엿,D-소비톨액(감미료) 5.4%,펙틴,EPA 및 DHA 함유 유지(조류추출오메가6),산도조절제1,산도조절제2,향료 5종,기타식용유지가공품,채소혼합농축액,레몬농축액,오렌지농축액,과채가공품,혼합제제[dl-α-토코페릴아세테이트, 변성전분(옥테닐호박산나트륨전분), 포도당시럽분말, 이산화규소],유화제,D-α-토코페롤",
        "영양정보(1일 섭취량 기준)": "1일 2회, 1회 1개",
        "섭취방법": "씹어서 섭취",
        "주의사항": "직사광선을 피하여 30℃ 이하에서 보관 및 개봉 후 가급적 빨리 드세요.",
    },
    "내츄럴플러스 어린이 알티지 오메가3 츄어블 오렌지맛 90캡슐 1통": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "정제어유(정제어유, d-토코페롤(혼합형), 노르웨이산),d-α-토코페롤,산화아연,베타카로틴혼합제제(베타카로틴, 옥수수유, 비타민E(dl-α-토코페롤)),비타민D3혼합제제(비타민D3, 가공유지(팜유), 비타민E(dl-α-토코페롤)),포도씨유(스페인산),말티톨(감미료),자일리톨(감미료),오렌지향(천연향료),오렌지향분말제제(오렌지주스, 덱스트린, 유당, 오렌지오일, 펙틴),레몬향(천연향료),요구르트향(향료),구연산,대두레시틴,효소처리스테비아(감미료)",
        "영양정보(1일 섭취량 기준)": "1일 1회, 1회 3캡슐",
        "섭취방법": "씹어서 섭취",
        "주의사항": "의약품(항응고제, 항혈소판제, 혈압강하제 등) 복용 시 전문가와 상담하십시오.개인에 따라 피부 관련 이상반응이 발생할 수 있습니다.고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.",
    },

    # ---- 그로우뉴(칼마디) ----
    "종근당 젤튼튼 어린이 키즈 칼슘 마그네슘 비타민D 아연 4박스": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "해조칼슘,산화마그네슘,산화아연,분말비타민D3혼합제제(비타민D3, 아라비아검, 설탕, 전분, 가공유지, 이산화규소, 비타민E),황산망간,유당혼합분말(유당, 덱스트린),포도당,D-소비톨(감미료),전지분유,히드록시프로필메틸셀룰로스,향료(밀크향분말),스테아린산마그네슘,크림분말향혼합제제(덱스트린, 프로필렌글리콜, 정제수, 주정, 아라비아검, 합성향료, 천연향료, 트리아세틴, 산도조절제, 카라멜수),효소처리스테비아(감미료),비트레드,자일리톨(감미료),황기추출분말",
        "영양정보(1일 섭취량 기준)": "1일 2회, 1회 1정",
        "섭취방법": "씹어서 섭취",
        "주의사항": "고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.용기 개봉 후 수분에 의해 정제 변색과 표면 반점이 일어날 우려가 있으므로 보관방법을 준수하시고 가급적 빨리 섭취하시기 바랍니다.",
    },
    "일양약품 칼슘 마그네슘 비타민D 비타민K 영양제 3개월분": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "해조칼슘,산화마그네슘,산화아연,황산망간,비타민D3혼합제제(비타민D3, 아라비아검, 옥수수전분, 설탕, 중쇄중성지방, 이산규소, 비타민E),비타민K1혼합제제(비타민K1, 아라비아검, 자당, 결정셀룰로스, 히드록시프로필메틸셀룰로스, 스테아린산마그네슘, 이산화규소),이산화규소,글리세린지방산에스테르,MSM,보스웰리아추출물분말",
        "영양정보(1일 섭취량 기준)": "1일 1정",
        "섭취방법": "씹어서 섭취",
        "주의사항": "직사광선이나 고온다습한 곳을 피해 서늘한 곳에 보관하십시오.",
    },
    "내츄럴플러스 어린이 칼슘 마그네슘 비타민D 아연 망간 츄어블 90정 6병": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "해조분말(해조분말, 옥수수전분/영국산),산화마그네슘,산화아연,비타민D3혼합제제(비타민D3, 아라비아검, 자당, 옥수수전분, 가공유지(팜유), 이산화규소, dl-α-토코페롤),황산망간,전지분유(원유:국산),무수결정포도당,자일리톨(감미료),식물성크림혼합분말[물엿, 식물성유지, 코코넛오일(외국산: 인도네시아, 필리핀, 말레이시아 등),농축우유단백분말(뉴질랜드산),제이인산칼륨,탈지유,제삼인산칼슘,대두레시틴],스테아린산마그네슘,향료(크림분말향),혼합제제(말토덱스트린, 프로필렌글리콜, 정제수, 주정, 아라비아검,합성향료, 천연향료, 트리아세틴, 초산에틸, 카라멜색소),이산화규소,효소처리스테비아(감미료),상어연골분말,유단백가수분해물,초유분말,폴리감마글루탐산",
        "영양정보(1일 섭취량 기준)": "1일 2회, 1회 1정",
        "섭취방법": "씹어서 섭취",
        "주의사항": "고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.",
    },

    # ---- 투데이D3(비타민D) ----
    "GNM 자연의품격 GNM 어린이 종합비타민 구미젤리 UP 1병": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "비타민C,셀레늄혼합제제(아셀렌산나트륨, 옥수수말토덱스트린),분말비타민E혼합제제(dl-α-토코페릴아세테이트, 변성전분, 포도당시럽분말, 이산화규소),니코틴산아미드,비타민D3혼합제제(비타민D3, 아라비아검, 수크로오스, 옥수수전분, 중쇄중성지방, 이산화규소, 비타민E),비타민A혼합제제(비타민A아세테이트, 포도당시럽분말, 옥수수전분, 아라비아검, 비타민E),비타민B6염산염,비오틴,물엿,설탕,정제수,젤라틴,D-소비톨액(감미료),산도조절제Ⅰ,향료(딸기향),펙틴,기타식용유지가공품,블랙캐럿농축액,산도조절제Ⅱ,글리세린지방산에스테르,딸기농축액,향료(혼합과일향)",
        "영양정보(1일 섭취량 기준)": "1일 1회, 1회 2구미",
        "섭취방법": "씹어서 섭취",
        "주의사항": "고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.신장질환이 있는 경우 섭취 전 전문가와 상담하십시오.손발 따끔거림, 작열감 또는 저림 등의 이상사례 발생 시 섭취를 중단하고 전문가와 상담하십시오.",
    },
    "일양약품 칼슘 마그네슘 비타민D 비타민K 영양제 9개월분": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "해조칼슘,산화마그네슘,산화아연,황산망간,비타민D3혼합제제(비타민D3, 아라비아검, 옥수수전분, 설탕, 중쇄중성지방, 이산규소, 비타민E),비타민K1혼합제제(비타민K1, 아라비아검, 자당, 결정셀룰로스, 히드록시프로필메틸셀룰로스,스테아린산마그네슘, 이산화규소),이산화규소,글리세린지방산에스테르,MSM,보스웰리아추출물분말",
        "영양정보(1일 섭취량 기준)": "1일 1회, 1회 1정",
        "섭취방법": "충분한 물과 함께 섭취",
        "주의사항": "습기에 민감한 제품이므로 개봉 후에는 뚜껑을 잘 닫아 보관하고, 습기가 차지 않을 수 있도록 주의하십시오.",
    },
    "드시모네 [베이비SET] 베이비스텝1 + 비타민D드롭스 10ml": {
        "식품의 유형": "건강기능식품",
        "원료명 및 함량": "프로바이오틱스 / 비타민D",
        "영양정보(1일 섭취량 기준)": "베이비스텝: 1일 1회, 1회 1포/ 비타민D: 1일 1회, 1회 2드롭(0.057ml)",
        "섭취방법": "직접 또는 분유에 섞어서 섭취, 이유식 등 음식과 함께 섭취",
        "주의사항": "냉장보관 등",
    },
}

# ── 데이터 ──────────────────────────────────────────────────────
product_specs = [
    # 카테고리, 제품명, 브랜드, 제형, 주요성분, 1일섭취량, 섭취방법, 경쟁포인트
    ("커큐민+",
     "[동가게PICK]SOVS 인퓨즈드워터+골든부스트 커큐민",
     "SOVS/동가게", "음료(750ml)",
     "바나바잎혼합추출액, 코로솔산",
     "1병(750ml)", "마셔서 섭취",
     "수분 보충+커큐민 복합 컨셉. 음료 형태 섭취 편의성"),
    ("커큐민+",
     "닥터린 하이퍼셀 수용성 커큐민 맥스 x5박스",
     "닥터린", "정제",
     "수용성커큐민, 흑후추추출물, 비타민C, 아연",
     "1정", "충분한 물과 함께",
     "수용성+흑후추 흡수율 소구. 복합 항산화 포뮬라"),
    ("커큐민+",
     "오허브 강황가루 강황분말 커큐민 1kg",
     "오허브", "분말",
     "강황뿌리줄기 100%(인도산)",
     "4~9g", "물에 섞거나 요리·음료와 함께",
     "식품형 대용량 가성비. 가격 앵커 가장 낮음"),

    ("지니어스뉴 (오메가3)",
     "휴럼 뮴 키즈 스마트 오메가3 구미 2박스",
     "휴럼", "구미(젤리)",
     "정제어유(멸치), DHA·EPA, 비타민C, 자일리톨",
     "2정", "씹어서 섭취",
     "구미 형태 섭취 편의. 멸치 기반 어류 오메가3"),
    ("지니어스뉴 (오메가3)",
     "네추럴라이즈 키즈 오메가3 꾸미 3박스",
     "네추럴라이즈", "구미(젤리)",
     "조류추출 오메가3(EPA+DHA), 펙틴, 채소·레몬·오렌지 농축액",
     "2개", "씹어서 섭취",
     "조류 유래 오메가3 → 비린내 없음 소구 가능. 식물성 젤라틴(펙틴)"),
    ("지니어스뉴 (오메가3)",
     "내츄럴플러스 어린이 알티지 오메가3 츄어블 90캡슐",
     "내츄럴플러스", "씹는 캡슐(츄어블)",
     "정제어유(노르웨이산), 비타민D3, 아연, 베타카로틴, 말티톨, 자일리톨",
     "3캡슐", "씹어서 섭취",
     "오메가3+비타민D+아연 복합. 노르웨이산 원료 신뢰 소구"),

    ("그로우뉴 (칼마디)",
     "종근당 젤튼튼 어린이 칼슘 마그네슘 비타민D 아연 4박스",
     "종근당", "정제(츄어블)",
     "해조칼슘, 산화마그네슘, 아연, 비타민D3, 황산망간, 황기추출분말",
     "2정", "씹어서 섭취",
     "종근당 브랜드 신뢰. 해조칼슘 기반. 밀크향으로 섭취 편의성"),
    ("그로우뉴 (칼마디)",
     "일양약품 칼슘 마그네슘 비타민D 비타민K 3개월분",
     "일양약품", "정제",
     "해조칼슘, 마그네슘, 비타민D3, 비타민K1, MSM, 보스웰리아추출물",
     "1정", "씹어서 섭취",
     "1일 1정 편의성. 비타민K1+MSM+보스웰리아 → 관절 복합 소구"),
    ("그로우뉴 (칼마디)",
     "내츄럴플러스 어린이 칼슘 마그네슘 비타민D 아연 망간 90정 6병",
     "내츄럴플러스", "정제(츄어블)",
     "해조분말, 마그네슘, 비타민D3, 아연, 망간, 상어연골, 초유분말, 폴리감마글루탐산",
     "2정", "씹어서 섭취",
     "성분 다양성 최대화(상어연골·초유·PGA). 대용량 구성"),

    ("투데이D3 (비타민D)",
     "GNM 어린이 종합비타민 구미젤리 UP 1병",
     "GNM", "구미(젤리)",
     "비타민A·C·D3·E·B6·셀레늄·비오틴, 딸기·블랙캐럿농축액",
     "2구미", "씹어서 섭취",
     "종합비타민 구미. 1회 2개로 여러 영양소 커버. 딸기향 기호성"),
    ("투데이D3 (비타민D)",
     "일양약품 칼슘 마그네슘 비타민D 비타민K 9개월분",
     "일양약품", "정제",
     "해조칼슘, 마그네슘, 비타민D3, 비타민K1, MSM, 보스웰리아추출물",
     "1정", "충분한 물과 함께",
     "9개월 대용량 → 단가 경쟁력. 칼슘·D3·K1 묶음 구성"),
    ("투데이D3 (비타민D)",
     "드시모네 [베이비SET] 베이비스텝1 + 비타민D드롭스 10ml",
     "드시모네", "드롭스+분말",
     "프로바이오틱스(유산균) + 비타민D",
     "1포+2드롭", "분유·이유식에 섞어 섭취",
     "유산균+비타민D 세트 → 영아 대상 복합 영양 세트 포지션"),
]

cols = ["카테고리", "제품명", "브랜드", "제형", "주요성분", "1일섭취량", "섭취방법", "경쟁포인트"]
df_spec = pd.DataFrame(product_specs, columns=cols)


# ── 스타일 ──────────────────────────────────────────────────────
CAT_COLORS = {
    "커큐민+":           "#D6E4F0",
    "지니어스뉴 (오메가3)": "#E8F5E9",
    "그로우뉴 (칼마디)":   "#F3E5F5",
    "투데이D3 (비타민D)":  "#FFF8E1",
}
 
def color_row(row):
    bg = CAT_COLORS.get(row["카테고리"], "#FFFFFF")
    return [f"background-color: {bg}"] * len(row)
 
styled = (
    df_spec.style
    .apply(color_row, axis=1)
    .set_properties(**{
        "font-size": "12px",
        "font-family": "Arial, sans-serif",
        "color": "#111111",
        "text-align": "left",
        "white-space": "pre-wrap",
        "vertical-align": "top",
        "border": "1px solid #d0d0d0",
        "padding": "6px 8px",
    })
    .set_properties(subset=["경쟁포인트"], **{
        "color": "#1F5C1F",
        "font-weight": "bold",
    })
    .set_table_styles([
        {"selector": "thead th", "props": [
            ("background-color", "#2F5496"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("font-size", "12px"),
            ("text-align", "center"),
            ("padding", "8px"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
        ]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "15px"),
            ("font-weight", "bold"),
            ("color", "#1F3864"),
            ("padding", "10px 0 6px 0"),
            ("text-align", "left"),
        ]},
    ])
    .set_caption("CJ온스타일 경쟁 제품 스펙 비교")
    .hide(axis="index")
)
 
display(styled)

카테고리,제품명,브랜드,제형,주요성분,1일섭취량,섭취방법,경쟁포인트
커큐민+,[동가게PICK]SOVS 인퓨즈드워터+골든부스트 커큐민,SOVS/동가게,음료(750ml),"바나바잎혼합추출액, 코로솔산",1병(750ml),마셔서 섭취,수분 보충+커큐민 복합 컨셉. 음료 형태 섭취 편의성
커큐민+,닥터린 하이퍼셀 수용성 커큐민 맥스 x5박스,닥터린,정제,"수용성커큐민, 흑후추추출물, 비타민C, 아연",1정,충분한 물과 함께,수용성+흑후추 흡수율 소구. 복합 항산화 포뮬라
커큐민+,오허브 강황가루 강황분말 커큐민 1kg,오허브,분말,강황뿌리줄기 100%(인도산),4~9g,물에 섞거나 요리·음료와 함께,식품형 대용량 가성비. 가격 앵커 가장 낮음
지니어스뉴 (오메가3),휴럼 뮴 키즈 스마트 오메가3 구미 2박스,휴럼,구미(젤리),"정제어유(멸치), DHA·EPA, 비타민C, 자일리톨",2정,씹어서 섭취,구미 형태 섭취 편의. 멸치 기반 어류 오메가3
지니어스뉴 (오메가3),네추럴라이즈 키즈 오메가3 꾸미 3박스,네추럴라이즈,구미(젤리),"조류추출 오메가3(EPA+DHA), 펙틴, 채소·레몬·오렌지 농축액",2개,씹어서 섭취,조류 유래 오메가3 → 비린내 없음 소구 가능. 식물성 젤라틴(펙틴)
지니어스뉴 (오메가3),내츄럴플러스 어린이 알티지 오메가3 츄어블 90캡슐,내츄럴플러스,씹는 캡슐(츄어블),"정제어유(노르웨이산), 비타민D3, 아연, 베타카로틴, 말티톨, 자일리톨",3캡슐,씹어서 섭취,오메가3+비타민D+아연 복합. 노르웨이산 원료 신뢰 소구
그로우뉴 (칼마디),종근당 젤튼튼 어린이 칼슘 마그네슘 비타민D 아연 4박스,종근당,정제(츄어블),"해조칼슘, 산화마그네슘, 아연, 비타민D3, 황산망간, 황기추출분말",2정,씹어서 섭취,종근당 브랜드 신뢰. 해조칼슘 기반. 밀크향으로 섭취 편의성
그로우뉴 (칼마디),일양약품 칼슘 마그네슘 비타민D 비타민K 3개월분,일양약품,정제,"해조칼슘, 마그네슘, 비타민D3, 비타민K1, MSM, 보스웰리아추출물",1정,씹어서 섭취,1일 1정 편의성. 비타민K1+MSM+보스웰리아 → 관절 복합 소구
그로우뉴 (칼마디),내츄럴플러스 어린이 칼슘 마그네슘 비타민D 아연 망간 90정 6병,내츄럴플러스,정제(츄어블),"해조분말, 마그네슘, 비타민D3, 아연, 망간, 상어연골, 초유분말, 폴리감마글루탐산",2정,씹어서 섭취,성분 다양성 최대화(상어연골·초유·PGA). 대용량 구성
투데이D3 (비타민D),GNM 어린이 종합비타민 구미젤리 UP 1병,GNM,구미(젤리),"비타민A·C·D3·E·B6·셀레늄·비오틴, 딸기·블랙캐럿농축액",2구미,씹어서 섭취,종합비타민 구미. 1회 2개로 여러 영양소 커버. 딸기향 기호성


---
---

# [자사몰 - 고도몰]

- 경쟁 제품과의 스펙 비교 및 차별화 분석을 위한 자사 제품 스펙 확인

In [74]:
# ── 입력 템플릿 ─────────────────────────────────────────────────
# 빈 문자열("")로 된 칸을 실제 스펙으로 교체하세요
lifebio_specs = [
    # ---- 커큐민+ ----
    {
        "카테고리":   "커큐민+",
        "제품명":     "커큐민+",           # 정확한 제품명으로 수정
        "브랜드":     "혜인서",
        "유형":       "기타식물성유지",                  # 예: 건강기능식품 / 기능성표시식품 / 일반식품
        "제형":       "정제",                  # 예: 정제 / 드롭스 / 분말 / 구미
        "주요성분":   "기타식물성유지[안틴플라TM메가큐민3/조류추출유지(미국산), 강황추출액{퀵큐민TM/강황추출액(노바솔커큐민/독일산)}]",                  # 원료명 및 함량 (핵심만)
        "함량기준":   "밀납, 올리브유(스페인산), 프로폴리스추출물분말(호주산), 흑후추추출물분말(인도산) / 할랄 인증, 코셔 인증 완료",                  # 예: 커큐민으로서 OOmg / 1일 O정
        "1일섭취량":  "1일 1회 2캡슐",                  # 예: 1정 / 2캡슐 / 5ml
        "섭취방법":   "물과 함께 섭취",                  # 예: 씹어서 섭취 / 분유에 섞어 섭취
        "주의사항":   "섭취 시 설사, 위통, 복부팽만, 소화불량, 위장관계 장애발생 시 섭취를 중단하십시오.개인에 따라 피부관련 이상반응이 발생할 수 있습니다.",
        "공식몰URL":  "https://m.phytonutri.kr/goods/goods_view.php?goodsNo=1000000251",                  # 상품 상세페이지 URL (선택)
    },

    # ---- 지니어스뉴 드롭스 ----
    {
        "카테고리":   "지니어스뉴 (오메가3)",
        "제품명":     "지니어스뉴 드롭스",
        "브랜드":     "그로우랩",
        "유형":       "기타식물성유지",
        "제형":       "드롭스(액상 오일)",       # 알고 있는 정보는 미리 채워둠
        "주요성분":   "DHA 200mg,ALA 200mg,인지질17μg,포스파티딜콜린3μg,L-세린95μg,비타민E 0.41mgα-TE 24%",
        "함량기준":   "올로메가3베이비스마트99.99%[EPA 및 DHA 함유유지{해조류추출디에이치에이오일, 고올레산해바라기유, 로즈마리추출물, d-토코페롤, L-아스코빌팔미테이트(산화방지제)/ (미국산),기타식물성유지{들깨쇠비름혼합추출물(들깨:국산,쇠비름:국산)},대두레시틴,d-α-토코페롤],L-세린",
        "1일섭취량":  "1일 섭취량 약 0.8~0.85 g / 12~24개월 유아",
        "섭취방법":   "이유식과 함께 섭취",
        "주의사항":   "섭취량 및 섭취방법을 준수. 원료 특성상 침전물이 생길 수 있으니, 흔들어 섭취.",
        "공식몰URL":  "https://m.phytonutri.kr/goods/goods_view.php?goodsNo=1000000281",
    },

    # ---- 지니어스뉴 톡캡스 ----
    {
        "카테고리":   "지니어스뉴 (오메가3)",
        "제품명":     "지니어스뉴 톡캡스",
        "브랜드":     "그로우랩",
        "유형":       "기타 식용유지가공품",
        "제형":       "연질캡슐(톡캡스) / 오일",
        "주요성분":   "DHA 150mg,ALA 150mg,인지질350μg,포스파티딜콜린80μg,비타민E 2.66mgα-TE 24%",
        "함량기준":   "조류DHA추출유지[미국산/해조류추출DHA오일, 고올레산 해바라기유, 로즈마리추출물, d-토코페롤(혼합형), L-아스 코빌팔미테이트(산화방지제)],들깨쇠비름혼합추출물(국산 /추출들깨유, 쇠비름혼합유), d-α-토코페롤, 유기농대두레 시틴 0.093%",
        "1일섭취량":  "1일 1회, 1회 1캡슐 (630mg) / 6~12개월 유아",
        "섭취방법":   "상단부의 손잡이를 돌려 제거 후 내용물을 짜서 섭취",
        "주의사항":   "섭취량 및 섭취방법을 준수. 원료 특성상 침전물이 생길 수 있으니, 흔들어 섭취.",
        "공식몰URL":  "https://m.phytonutri.kr/goods/goods_view.php?goodsNo=1000000368",
    },

    # ---- 그로우뉴 칼슘·마그네슘 ----
    {
        "카테고리":   "그로우뉴 (칼마디)",
        "제품명":     "그로우뉴 아이 유기농칼슘&마그네슘 비타민D·K2·망간",
        "브랜드":     "그로우랩",
        "유형":       "캔디류",
        "제형":       "정제",
        "주요성분":   "칼슘 210 mg, 마그네슘 7.6 mg, 망간 2 mg, 비타민D 10 ㎍, 비타민K 30 ㎍",
        "함량기준":   "유기농포도당 41.014 %, 유기농아가베추출분말(멕시코산) 27 %, 유기농식용난각분말(벨기에산) 12.821 %, 유기농코코아분말(네덜란드산) 10 %, 유기농쌀겨추출물 5%, 쌀발효분말, 초콜렛향분말혼합제제(분말결정포도당, 프로필렌글리콜, 카라멜시럽, 향료), 건조효모(망간), 효소처리스테비아(감미료), 바실러스나토균농축분말, 건조효모(비타민D3) 알류(가금류) 함유",
        "1일섭취량":  "1일 1회, 1회 3정",
        "섭취방법":   "씹어서 섭취",
        "주의사항":   "고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.항응고제 등 복용 시 전문가와 상담하십시오.",
        "공식몰URL":  "https://m.phytonutri.kr/goods/goods_view.php?goodsNo=1000000397",
    },

    # ---- 투데이D3 ----
    {
        "카테고리":   "투데이D3 (비타민D)",
        "제품명":     "투데이D3",           # 정확한 제품명으로 수정
        "브랜드":     "파이토뉴트리",
        "유형":       "기타 식용유지가공품",
        "제형":       "연질캡슐(톡캡스) / 오일",
        "주요성분":   "비타민D 10㎍ (400IU)",
        "함량기준":   "유기농압착호두오일(독일산) 75.938 %, 오르지닉D3[유기농압착호두오일(독일산) 92.9195 %, 유기농발효건조효모D(미국산) 7.0805 %], 유기농밀납 10.4 %, 유기농대두레시틴(미국산) 2.9 % ※ 캡슐기제: 히드록시프로필전분, 글리세린, 카라기난 - 투데이디 balm : 유기농압착호두오일 (독일산) 78.838%, 오르지닉D3 [유기농압착호두오일] (독일산) 92.9195%, 유기농발효건조효모D(미국산) 7.0805%], 유기농밀납 10.4% / ※ 캡슐기제 : 히드록시프로필전분,글리세린,카라기난",
        "1일섭취량":  "1일 1회, 1회 1캡슐",
        "섭취방법":   "상단부의 손잡이를 돌려 제거 후 내용물을 짜서 섭취",
        "주의사항":   "개봉 후 뚜껑을 닫아 실온 보관",
        "공식몰URL":  "https://m.phytonutri.kr/goods/goods_view.php?goodsNo=1000000329",
    },

    # 제품 추가가 필요하면 위 블록을 복사해서 아래에 붙여넣으세요
]

# ── 미입력 현황 확인 ────────────────────────────────────────────
df_lifebio_spec = pd.DataFrame(lifebio_specs)
empty_check = df_lifebio_spec.replace("", pd.NA).isna()
empty_cols_per_product = empty_check.sum(axis=1)

print("=== 미입력 항목 현황 ===")
for i, row in df_lifebio_spec.iterrows():
    empties = [col for col in df_lifebio_spec.columns if row[col] == ""]
    if empties:
        print(f"  [{row['제품명']}] 미입력: {', '.join(empties)}")
    else:
        print(f"  [{row['제품명']}] ✅ 완료")

# ── 미리보기 ────────────────────────────────────────────────────
CAT_COLORS = {
    "커큐민+":               "#D6E4F0",
    "지니어스뉴 (오메가3)":   "#E8F5E9",
    "그로우뉴 (칼마디)":      "#F3E5F5",
    "투데이D3 (비타민D)":     "#FFF8E1",
}

def highlight_empty(val):
    return "background-color: #FFE0E0; color: #9C2700;" if val == "" else ""

def color_row(row):
    bg = CAT_COLORS.get(row["카테고리"], "#FFFFFF")
    return [f"background-color: {bg}"] * len(row)

styled = (
    df_lifebio_spec.drop(columns=["공식몰URL"])   # URL은 표에서 숨김
    .style
    .apply(color_row, axis=1)
    .applymap(highlight_empty)                     # 빈 칸 빨간 강조
    .set_properties(**{
        "font-size":    "12px",
        "font-family":  "Arial, sans-serif",
        "color":        "#111111",
        "text-align":   "left",
        "white-space":  "pre-wrap",
        "vertical-align": "top",
        "border":       "1px solid #d0d0d0",
        "padding":      "6px 8px",
    })
    .set_table_styles([
        {"selector": "thead th", "props": [
            ("background-color", "#2F5496"),
            ("color", "white"),
            ("font-weight", "bold"),
            ("font-size", "12px"),
            ("text-align", "center"),
            ("padding", "8px"),
        ]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        {"selector": "caption", "props": [
            ("caption-side", "top"),
            ("font-size", "15px"),
            ("font-weight", "bold"),
            ("color", "#1F3864"),
            ("padding", "10px 0 6px 0"),
            ("text-align", "left"),
        ]},
    ])
    .set_caption("라이프앤바이오 자사 제품 스펙 (빨간 셀 = 미입력)")
    .hide(axis="index")
)

display(styled)

=== 미입력 항목 현황 ===
  [커큐민+] ✅ 완료
  [지니어스뉴 드롭스] ✅ 완료
  [지니어스뉴 톡캡스] ✅ 완료
  [그로우뉴 아이 유기농칼슘&마그네슘 비타민D·K2·망간] ✅ 완료
  [투데이D3] ✅ 완료


C:\Users\jm\AppData\Local\Temp\ipykernel_3004\2884110247.py:114: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_empty)                     # 빈 칸 빨간 강조


카테고리,제품명,브랜드,유형,제형,주요성분,함량기준,1일섭취량,섭취방법,주의사항
커큐민+,커큐민+,혜인서,기타식물성유지,정제,"기타식물성유지[안틴플라TM메가큐민3/조류추출유지(미국산), 강황추출액{퀵큐민TM/강황추출액(노바솔커큐민/독일산)}]","밀납, 올리브유(스페인산), 프로폴리스추출물분말(호주산), 흑후추추출물분말(인도산) / 할랄 인증, 코셔 인증 완료",1일 1회 2캡슐,물과 함께 섭취,"섭취 시 설사, 위통, 복부팽만, 소화불량, 위장관계 장애발생 시 섭취를 중단하십시오.개인에 따라 피부관련 이상반응이 발생할 수 있습니다."
지니어스뉴 (오메가3),지니어스뉴 드롭스,그로우랩,기타식물성유지,드롭스(액상 오일),"DHA 200mg,ALA 200mg,인지질17μg,포스파티딜콜린3μg,L-세린95μg,비타민E 0.41mgα-TE 24%","올로메가3베이비스마트99.99%[EPA 및 DHA 함유유지{해조류추출디에이치에이오일, 고올레산해바라기유, 로즈마리추출물, d-토코페롤, L-아스코빌팔미테이트(산화방지제)/ (미국산),기타식물성유지{들깨쇠비름혼합추출물(들깨:국산,쇠비름:국산)},대두레시틴,d-α-토코페롤],L-세린",1일 섭취량 약 0.8~0.85 g / 12~24개월 유아,이유식과 함께 섭취,"섭취량 및 섭취방법을 준수. 원료 특성상 침전물이 생길 수 있으니, 흔들어 섭취."
지니어스뉴 (오메가3),지니어스뉴 톡캡스,그로우랩,기타 식용유지가공품,연질캡슐(톡캡스) / 오일,"DHA 150mg,ALA 150mg,인지질350μg,포스파티딜콜린80μg,비타민E 2.66mgα-TE 24%","조류DHA추출유지[미국산/해조류추출DHA오일, 고올레산 해바라기유, 로즈마리추출물, d-토코페롤(혼합형), L-아스 코빌팔미테이트(산화방지제)],들깨쇠비름혼합추출물(국산 /추출들깨유, 쇠비름혼합유), d-α-토코페롤, 유기농대두레 시틴 0.093%","1일 1회, 1회 1캡슐 (630mg) / 6~12개월 유아",상단부의 손잡이를 돌려 제거 후 내용물을 짜서 섭취,"섭취량 및 섭취방법을 준수. 원료 특성상 침전물이 생길 수 있으니, 흔들어 섭취."
그로우뉴 (칼마디),그로우뉴 아이 유기농칼슘&마그네슘 비타민D·K2·망간,그로우랩,캔디류,정제,"칼슘 210 mg, 마그네슘 7.6 mg, 망간 2 mg, 비타민D 10 ㎍, 비타민K 30 ㎍","유기농포도당 41.014 %, 유기농아가베추출분말(멕시코산) 27 %, 유기농식용난각분말(벨기에산) 12.821 %, 유기농코코아분말(네덜란드산) 10 %, 유기농쌀겨추출물 5%, 쌀발효분말, 초콜렛향분말혼합제제(분말결정포도당, 프로필렌글리콜, 카라멜시럽, 향료), 건조효모(망간), 효소처리스테비아(감미료), 바실러스나토균농축분말, 건조효모(비타민D3) 알류(가금류) 함유","1일 1회, 1회 3정",씹어서 섭취,고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담하십시오.항응고제 등 복용 시 전문가와 상담하십시오.
투데이D3 (비타민D),투데이D3,파이토뉴트리,기타 식용유지가공품,연질캡슐(톡캡스) / 오일,비타민D 10㎍ (400IU),"유기농압착호두오일(독일산) 75.938 %, 오르지닉D3[유기농압착호두오일(독일산) 92.9195 %, 유기농발효건조효모D(미국산) 7.0805 %], 유기농밀납 10.4 %, 유기농대두레시틴(미국산) 2.9 % ※ 캡슐기제: 히드록시프로필전분, 글리세린, 카라기난 - 투데이디 balm : 유기농압착호두오일 (독일산) 78.838%, 오르지닉D3 [유기농압착호두오일] (독일산) 92.9195%, 유기농발효건조효모D(미국산) 7.0805%], 유기농밀납 10.4% / ※ 캡슐기제 : 히드록시프로필전분,글리세린,카라기난","1일 1회, 1회 1캡슐",상단부의 손잡이를 돌려 제거 후 내용물을 짜서 섭취,개봉 후 뚜껑을 닫아 실온 보관


### 0. 자사 vs 경쟁사 스펙 비교

In [75]:
# ================================================================
# [자사 vs 경쟁사] 카테고리별 스펙 비교표
#
# 사전 조건:
#   - df_lifebio_spec   : 자사 제품 스펙 DataFrame (lifebio_spec_input 셀)
#   - df_spec           : CJ온스타일 경쟁 제품 DataFrame (cj_spec_comparison_notebook 셀)
#   - df_coupang        : 쿠팡 경쟁 제품 DataFrame (coupang_spec_display 셀)
# ================================================================
import pandas as pd
from IPython.display import display, HTML

# ── 비교할 공통 컬럼 ─────────────────────────────────────────────
SHOW_COLS = ["제품명", "브랜드", "제형", "주요성분", "1일섭취량", "섭취방법"]

CAT_ORDER = ["커큐민+", "지니어스뉴 (오메가3)", "그로우뉴 (칼마디)", "투데이D3 (비타민D)"]

CAT_COLORS = {
    "커큐민+":               "#D6E4F0",
    "지니어스뉴 (오메가3)":   "#E8F5E9",
    "그로우뉴 (칼마디)":      "#F3E5F5",
    "투데이D3 (비타민D)":     "#FFF8E1",
}

SOURCE_STYLES = {
    "자사몰":   {"bg": "#1F3864", "label": "🏠 자사몰"},
    "쿠팡":     {"bg": "#E8390E", "label": "🛒 쿠팡"},
    "CJ온스타일": {"bg": "#005BAC", "label": "📺 CJ온스타일"},
}

# ── 각 DataFrame에 출처 컬럼 추가 및 컬럼 통일 ───────────────────
def prep(df, source):
    d = df.copy()
    # 쿠팡 df는 컬럼명이 약간 다를 수 있으므로 rename
    rename_map = {
        "식품유형": "유형",
        "생산자/소재지": "생산자",
        "원료명 및 함량": "주요성분",
        "영양정보(1일 섭취량 기준)": "1일섭취량",
    }
    d = d.rename(columns={k: v for k, v in rename_map.items() if k in d.columns})
    d["출처"] = source
    # 없는 컬럼은 빈 문자열로
    for col in SHOW_COLS + ["카테고리", "출처"]:
        if col not in d.columns:
            d[col] = ""
    return d[["카테고리", "출처"] + SHOW_COLS]

df_self   = prep(df_lifebio_spec, "자사몰")
df_cj_p   = prep(df_spec,         "CJ온스타일")
df_coup_p = prep(df_coupang,      "쿠팡")

# ── 합치기 ───────────────────────────────────────────────────────
df_all = pd.concat([df_self, df_cj_p, df_coup_p], ignore_index=True)
df_all = df_all[df_all["카테고리"].isin(CAT_ORDER)]

# ── 카테고리별로 분리해서 출력 ────────────────────────────────────
def source_badge(source):
    s = SOURCE_STYLES.get(source, {"bg": "#888", "label": source})
    return (
        f'<span style="background:{s["bg"]};color:white;'
        f'padding:2px 7px;border-radius:4px;font-size:10px;'
        f'font-weight:bold;white-space:nowrap">{s["label"]}</span>'
    )

def make_cat_table(cat, df_cat):
    rows_html = ""
    bg = CAT_COLORS.get(cat, "#FFFFFF")

    # 자사몰 먼저, 그 다음 쿠팡/CJ 순서
    source_order = ["자사몰", "쿠팡", "CJ온스타일"]
    df_cat = df_cat.copy()
    df_cat["_order"] = df_cat["출처"].map({s: i for i, s in enumerate(source_order)}).fillna(99)
    df_cat = df_cat.sort_values("_order").drop(columns="_order")

    for _, row in df_cat.iterrows():
        is_self = row["출처"] == "자사몰"
        row_style = (
            f'background:{bg};font-weight:bold;border-left:4px solid #1F3864;'
            if is_self else
            f'background:{"#FFFFFF"};'
        )
        badge = source_badge(row["출처"])
        cells = f'<td style="padding:7px 10px;border:1px solid #d0d0d0;white-space:nowrap">{badge}</td>'
        for col in SHOW_COLS:
            val = str(row.get(col, ""))
            style = "padding:6px 9px;border:1px solid #d0d0d0;font-size:11px;vertical-align:top;white-space:pre-wrap;color:#111111;"
            if is_self:
                style += "font-weight:bold;"
            cells += f'<td style="{style}">{val}</td>'
        rows_html += f'<tr style="{row_style}">{cells}</tr>'

    header_cells = '<th style="background:#2F5496;color:white;padding:7px 10px;font-size:11px;white-space:nowrap">출처</th>'
    for col in SHOW_COLS:
        header_cells += f'<th style="background:#2F5496;color:white;padding:7px 10px;font-size:11px;white-space:nowrap">{col}</th>'

    html = f"""
    <div style="margin-bottom:28px;">
      <div style="font-size:14px;font-weight:bold;color:#1F3864;
                  border-left:5px solid {list(CAT_COLORS.values())[CAT_ORDER.index(cat)]};
                  padding-left:10px;margin-bottom:8px;">
        ▶ {cat}
      </div>
      <table style="border-collapse:collapse;width:100%;font-family:Arial,sans-serif;">
        <thead><tr>{header_cells}</tr></thead>
        <tbody>{rows_html}</tbody>
      </table>
    </div>
    """
    return html

# ── 전체 렌더링 ──────────────────────────────────────────────────
title_html = """
<div style="font-size:17px;font-weight:bold;color:#1F3864;
            border-bottom:3px solid #2F5496;padding-bottom:8px;margin-bottom:20px;">
  자사몰(고도몰) vs 쿠팡 &amp; CJ온스타일 — 카테고리별 제품 스펙 비교
  <span style="font-size:11px;font-weight:normal;color:#555;margin-left:12px;">
    🏠 자사몰(굵은 글씨·파란 테두리)  🛒 쿠팡  📺 CJ온스타일
  </span>
</div>
"""

body_html = ""
for cat in CAT_ORDER:
    df_cat = df_all[df_all["카테고리"] == cat]
    if df_cat.empty:
        continue
    body_html += make_cat_table(cat, df_cat)

display(HTML(title_html + body_html))

출처,제품명,브랜드,제형,주요성분,1일섭취량,섭취방법
🏠 자사몰,커큐민+,혜인서,정제,"기타식물성유지[안틴플라TM메가큐민3/조류추출유지(미국산), 강황추출액{퀵큐민TM/강황추출액(노바솔커큐민/독일산)}]",1일 1회 2캡슐,물과 함께 섭취
🛒 쿠팡,센트휴 - 수용성 커큐민 바이오페인2X,,,"수용성 강황추출물분말(인도산) 49.99%,치커리뿌리추출물분말(식이섬유 80% 이상/벨기에산),덱스트린,흑후추추출물(바이오페린 95% 이상/인도산),글루칸-30(덴마크산),캐롭분말 다크(스페인산),버섯혼합추출분말,영지버섯추출분말,강황추출분말,이산화규소,스테아린산마그네슘",1일 1회 1~2정,물과 함께 섭취
🛒 쿠팡,순수채움 - 수용성 커큐민 맥시멈,,,"수용성 커큐민 추출분말(수용성 커큐민 추출물(인도),난소화성말토덱스트린),포도당,생강추출분말(국내산),27종 과일야채혼합분말(국내제조),보스웰리아추출물분말(인도),비타민C,퀘르세틴,흑후추추출분말",1일 1회 1~2정,물과 함께 섭취
🛒 쿠팡,가온담음 - 페라큐민 강황 수용성 커큐민,,,"수용성 커큐민 추출분말(수용성 커큐민 추출분말[강황추출물분말(인도 산)(강황뿌리줄기) 난소화성말토덱스 트린],포도당,생강추출(국내산),27종과일야채혼합농축분말[세븐베리 농축액[블랙베리농축액(독일)] 야채 혼합농축액(독일산)],보스웰리아추출 물분말(인도산),비타민C,게르세틴,흑후추추출분말 토마토 함유",1일 1회 1정,물과 함께 섭취
📺 CJ온스타일,[동가게PICK]SOVS 인퓨즈드워터+골든부스트 커큐민,SOVS/동가게,음료(750ml),"바나바잎혼합추출액, 코로솔산",1병(750ml),마셔서 섭취
📺 CJ온스타일,닥터린 하이퍼셀 수용성 커큐민 맥스 x5박스,닥터린,정제,"수용성커큐민, 흑후추추출물, 비타민C, 아연",1정,충분한 물과 함께
📺 CJ온스타일,오허브 강황가루 강황분말 커큐민 1kg,오허브,분말,강황뿌리줄기 100%(인도산),4~9g,물에 섞거나 요리·음료와 함께
출처,제품명,브랜드,제형,주요성분,1일섭취량,섭취방법
🏠 자사몰,지니어스뉴 드롭스,그로우랩,드롭스(액상 오일),"DHA 200mg,ALA 200mg,인지질17μg,포스파티딜콜린3μg,L-세린95μg,비타민E 0.41mgα-TE 24%",1일 섭취량 약 0.8~0.85 g / 12~24개월 유아,이유식과 함께 섭취
🏠 자사몰,지니어스뉴 톡캡스,그로우랩,연질캡슐(톡캡스) / 오일,"DHA 150mg,ALA 150mg,인지질350μg,포스파티딜콜린80μg,비타민E 2.66mgα-TE 24%","1일 1회, 1회 1캡슐 (630mg) / 6~12개월 유아",상단부의 손잡이를 돌려 제거 후 내용물을 짜서 섭취
